In [1]:
import os

# =========================================================
# USER CONFIGURATION — CHANGE ONLY VALUES HERE
# =========================================================

# OpenRouter
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

# Model settings
MODEL_ID = "anthropic/claude-sonnet-4"  # Vision model for image classification
OPTIMIZER_MODEL_ID = "openai/gpt-4o-mini"  # Cheaper text-only model for prompt optimization (~95% cost savings)
MODEL_TEMPERATURE = 0.1  # Vision model temperature (for evaluation during optimization)
FINAL_EVAL_TEMPERATURE = 0.1  # Temperature for final evaluation (more realistic, matches production conditions)
OPTIMIZER_TEMPERATURE = 0.0  # Optimizer temperature (lower for stability, 0.0-0.1 recommended)

# Optimization limits
MIN_PASS_IMAGES_REQUIRED = 3        # safety guard
OPTIMIZATION_ITERATIONS = 5        # Optimized for cost efficiency with prompt caching
EVALUATION_ONLY = False             # If True, run only baseline evaluation (iteration 0), skip optimization
TARGET_RECALL = 95.0                # Target recall percentage
TARGET_PRECISION = 95.0             # Target precision percentage
EARLY_STOP_THRESHOLD = 95.0         # Stop early if both metrics reach this
REFERENCE_EXAMPLES_THRESHOLD = 90.0  # Only use reference examples if precision OR recall < this (saves ~30% on image calls)

# Hard set optimization (saves ~35-40% on costs)
USE_HARD_SET_OPTIMIZATION = False  # Optimize only on hard cases (FN+FP+random), then evaluate on all (DISABLED - increases cost)
HARD_SET_SIZE = 15  # Target size for hard set (will include all FN+FP + random to reach this size)
HARD_SET_RANDOM_COUNT = 3  # Number of random PASS/FAIL to add for diversity
USE_DETERMINISTIC_DECISION = False  # Use deterministic decision from rules (False = use model decision, more flexible)

# Cost optimization settings
PERSISTENT_CACHE_ENABLED = True  # Save cache to disk (persists across runs, saves API costs)
CACHE_DIR = "./.api_cache"  # Directory for persistent cache files
IMAGE_MAX_SIZE_MB = 3.5  # Reduced from 4.5MB for more aggressive compression (saves image token costs)
CHEAPER_VISION_MODEL = "anthropic/claude-3-haiku"  # Cheaper model for simple/clear cases
USE_CHEAPER_MODEL_THRESHOLD = 0.95  # Use cheaper model if previous iteration confidence > 95%

# Request behavior
OPENROUTER_TIMEOUT_SECONDS = 120
OPENROUTER_RETRIES = 3

# Parallel processing
# Number of concurrent API calls (higher = faster but may hit rate limits)
# Recommended values:
#   - 3-5: Safe, works with most APIs
#   - 5-10: Faster, good for OpenRouter/Claude
#   - 10+: Very fast, but may hit rate limits (check your API provider's limits)
MAX_WORKERS = 5

# Data / storage
PROJECTS_DIR_PATH = "./projects"

# =========================================================
# DEFAULT PROMPTS (Edit these to customize base prompts)
# =========================================================

# System prompt (EDIT THIS)
DEFAULT_SYSTEM_PROMPT = """
You are a micromobility parking enforcement officer. 
Analyze this parking photo of a shared e-scooter or bicycle.
Rate it as PASS or FAIL and provide feedback accordingly. Condense your feedback into a single, short sentence. 

 If FAIL, make it actionable so the rider knows what they need to improve. 
""".strip()

# Base user prompt (EDIT THIS - Easy access in first cell)
DEFAULT_BASE_USER_PROMPT = """

Apply the following rules:

Analyze this parking photo according to the rules below and decide whether the vehicle is correctly parked.

Evaluate only the four vehicle types used in this market:
• Spin scooters (black body with orange accents)
• Spin e-bikes (orange frame)
• Bird scooters (gray body with light blue accents)
• Bird e-bikes (blue body with black accent)

Parking rules
• The vehicle must be mostly visible in the photo.
• The vehicle must not be lying on its side or fully collapsed on the ground.
• A vehicle that is standing on its wheels, including when leaning on a kickstand or slightly tilted due to uneven ground, should be considered upright.
• The vehicle must not be held or supported by a person.
• The vehicle must not clearly block the pedestrian walking path and not be in the middle of the walking path, so that a pedestrian should be able to pass naturally
• Parking on sidewalks is allowed only if the vehicle is fully to the side and leaves a clearly usable pedestrian path.
• Parking in bike racks or the furniture zone is allowed.
• A vehicle is considered far enough from the main sidewalk if a pedestrian can walk naturally on the sidewalk without stepping onto dirt, grass, gravel, or changing direction.
• If the vehicle is positioned along the edge of the sidewalk or in the furniture zone and a pedestrian can obviously pass without changing direction, the result is PASS.
• Parking on vegetation (grass or planted areas) is allowed, as long as it does not block a pedestrian walking path.
• Parking on black gravel strip areas directly adjacent to the road or sidewalk is not allowed. 
• Parking on other gravel colors, including white and red, are allowed.
• Parking on gravel when the area is wide and open that does not function as a buffer from the road is allowed.

• Peromrm these checks for all our vehicle tyeps.
• If there are multiple vehicles of our brands, it is enough for one vehicle to PASS that the result is PASS.


""".strip()

# =========================================================


In [2]:
# ========================================================= 
# DEFAULT PROMPTS MOVED TO FIRST CELL (Cell 0) FOR EASY ACCESS
# =========================================================
# 
# The default prompts are now in the first code cell (configuration cell)
# for easy editing. They are defined as:
# - DEFAULT_SYSTEM_PROMPT
# - DEFAULT_BASE_USER_PROMPT
#
# Edit them in Cell 0 (the configuration cell at the top)


In [3]:
# =========================================================
# STREAMLIT UI (Recommended - Better file upload handling!)
# =========================================================
# To use Streamlit (RECOMMENDED for better file uploads):
#   1. Make sure streamlit is installed: pip install streamlit
#   2. Run from terminal: streamlit run app_streamlit.py
#   3. Or from Jupyter: !streamlit run app_streamlit.py
#
# Streamlit handles file uploads much better than Gradio and avoids
# the ClientDisconnect errors you've been experiencing.
#
# The Streamlit app is in: app_streamlit.py

In [4]:
import os
import zipfile
import tempfile
import shutil
import json
import base64
import mimetypes
import time
import logging
import threading
import hashlib
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import gradio as gr
import pandas as pd
from PIL import Image

# =========================================================
# LOGGING SETUP
# =========================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# =========================================================
# USER CONFIG — CHANGE ONLY HERE
# =========================================================
# Note: Temperature settings are defined in Cell 0 (first cell)
# MODEL_TEMPERATURE, FINAL_EVAL_TEMPERATURE, and OPTIMIZER_TEMPERATURE are set there

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
MODEL_ID = "anthropic/claude-sonnet-4"  # Vision model for image classification
OPTIMIZER_MODEL_ID = "openai/gpt-4o-mini"  # Cheaper text-only model for prompt optimization (~95% cost savings)

OPTIMIZATION_ITERATIONS = 5         # Optimized for cost efficiency with prompt caching
EVALUATION_ONLY = False             # If True, run only baseline evaluation (iteration 0), skip optimization
MIN_PASS_IMAGES_REQUIRED = 3         # safety guard so recall isn't meaningless
TARGET_RECALL = 95.0                 # Target recall percentage
TARGET_PRECISION = 95.0              # Target precision percentage
EARLY_STOP_THRESHOLD = 95.0          # Stop early if both metrics reach this
REFERENCE_EXAMPLES_THRESHOLD = 90.0  # Only use reference examples if precision OR recall < this (saves ~30% on image calls)

# Hard set optimization (saves ~35-40% on costs)
USE_HARD_SET_OPTIMIZATION = False  # Optimize only on hard cases (FN+FP+random), then evaluate on all (DISABLED - increases cost)
HARD_SET_SIZE = 15  # Target size for hard set (will include all FN+FP + random to reach this size)
HARD_SET_RANDOM_COUNT = 3  # Number of random PASS/FAIL to add for diversity
USE_DETERMINISTIC_DECISION = False  # Use deterministic decision from rules (False = use model decision, more flexible)

# Cost optimization settings
PERSISTENT_CACHE_ENABLED = True  # Save cache to disk (persists across runs, saves API costs)
CACHE_DIR = "./.api_cache"  # Directory for persistent cache files
IMAGE_MAX_SIZE_MB = 3.5  # Reduced from 4.5MB for more aggressive compression (saves image token costs)
CHEAPER_VISION_MODEL = "anthropic/claude-3-haiku"  # Cheaper model for simple/clear cases
USE_CHEAPER_MODEL_THRESHOLD = 0.95  # Use cheaper model if previous iteration confidence > 95%

OPENROUTER_TIMEOUT_SECONDS = 120
OPENROUTER_RETRIES = 3

# Global stop event for cancellation
stop_event = threading.Event()
# =========================================================


# -------------------- HTTP helper (retries) --------------------

def post_with_retries(
    url: str,
    headers: dict,
    payload: dict,
    timeout_s: int = OPENROUTER_TIMEOUT_SECONDS,
    retries: int = OPENROUTER_RETRIES,
) -> dict:
    """
    POST with retries + exponential backoff.
    Returns response.json() on success.
    Raises RuntimeError on final failure.
    """
    logger.info(f"Making request to {url} (timeout={timeout_s}s, retries={retries})")
    last_err: Optional[str] = None
    for attempt in range(1, retries + 1):
        try:
            logger.debug(f"Attempt {attempt}/{retries}")
            r = requests.post(url, headers=headers, json=payload, timeout=timeout_s)
            if r.status_code == 200:
                data = r.json()
                # Check for provider errors in the response (status 200 but error in body)
                if "error" in data:
                    error_info = data.get("error", {})
                    error_msg = error_info.get("message", "Unknown error")
                    # Try to extract more details from metadata
                    if "metadata" in error_info and "raw" in error_info["metadata"]:
                        try:
                            import json as json_module
                            raw_error = json_module.loads(error_info["metadata"]["raw"])
                            if "error" in raw_error and "message" in raw_error["error"]:
                                error_msg = raw_error["error"]["message"]
                        except:
                            pass
                    last_err = f"Provider error: {error_msg}"
                    logger.warning(f"Request returned error in response: {last_err}")
                else:
                    logger.info(f"Request successful on attempt {attempt}")
                    return data
            else:
                last_err = f"HTTP {r.status_code}: {r.text[:600]}"
                logger.warning(f"Request failed with status {r.status_code}: {last_err}")
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout, BrokenPipeError) as e:
            last_err = f"Connection error: {type(e).__name__}: {str(e)}"
            logger.warning(f"Request connection error on attempt {attempt}: {e}")
        except Exception as e:
            last_err = str(e)
            logger.warning(f"Request exception on attempt {attempt}: {type(e).__name__}: {e}")

        if attempt < retries:
            wait_time = 2 ** (attempt - 1)
            logger.info(f"Waiting {wait_time}s before retry...")
            time.sleep(wait_time)

    error_msg = f"OpenRouter request failed after {retries} attempts. Last error: {last_err}"
    logger.error(error_msg)
    raise RuntimeError(error_msg)


# -------------------- CSV and base photos --------------------

def load_base_photos_from_csv(market: str, csv_path: str = "ai photo review - examples.csv") -> Tuple[List[Dict[str, Any]], str]:
    """
    Load base photos from CSV file filtered by market (case-insensitive).
    
    Args:
        market: Market name to filter by (case-insensitive)
        csv_path: Path to CSV file
    
    Returns:
        Tuple of (list of photo dicts, error_message)
        Each photo dict has: image_url, evaluation_status, evaluation_message, market
        If error_message is not empty, the list will be empty.
    """
    if not market or not market.strip():
        return [], ""
    
    market = market.strip()
    
    # Simplified: Only check current directory and one level up (no recursive search)
    possible_paths = [
        csv_path,  # Current directory
        os.path.join("..", csv_path),  # Parent directory (one level up)
    ]
    
    csv_file_path = None
    for path in possible_paths:
        abs_path = os.path.abspath(path)
        if os.path.exists(abs_path) and os.path.isfile(abs_path):
            csv_file_path = abs_path
            break
    
    if not csv_file_path:
        return [], f"CSV file not found: {csv_path}. Please ensure the file exists in the project directory."
    
    try:
        # Load CSV
        df = pd.read_csv(csv_file_path)
        logger.info(f"Loaded CSV file: {csv_file_path} ({len(df)} rows)")
        
        # Check required columns
        required_cols = ['market', 'image_url', 'evaluation_status', 'evaluation_message']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            return [], f"CSV file missing required columns: {', '.join(missing_cols)}. Found columns: {', '.join(df.columns)}"
        
        # Filter by market (case-insensitive)
        df_filtered = df[df['market'].astype(str).str.upper().str.strip() == market.upper().strip()]
        
        if len(df_filtered) == 0:
            # Get unique markets for helpful error message
            unique_markets = df['market'].astype(str).str.strip().unique()[:10]
            return [], f"Market '{market}' not found in CSV file. Available markets (sample): {', '.join(unique_markets)}"
        
        # Filter out rows with empty or missing image_url
        df_filtered = df_filtered[df_filtered['image_url'].notna() & (df_filtered['image_url'].astype(str).str.strip() != "")]
        
        if len(df_filtered) == 0:
            return [], f"Market '{market}' found but has no photos with valid image_url"
        
        # Convert to list of dicts
        base_photos = []
        for _, row in df_filtered.iterrows():
            status = str(row['evaluation_status']).strip().upper()
            if status not in ['PASS', 'FAIL']:
                logger.warning(f"Skipping row with invalid evaluation_status: {status}")
                continue
            
            base_photos.append({
                'image_url': str(row['image_url']).strip(),
                'evaluation_status': status,
                'evaluation_message': str(row['evaluation_message']) if pd.notna(row['evaluation_message']) else "",
                'market': str(row['market']).strip()
            })
        
        logger.info(f"Loaded {len(base_photos)} base photos from CSV for market '{market}'")
        
        # Also check for local folders matching the market name
        local_photos = []
        market_folder = None
        
        # Check current directory and one level up for folders matching market name (case-insensitive)
        search_dirs = [".", ".."]
        for search_dir in search_dirs:
            abs_search_dir = os.path.abspath(search_dir)
            if os.path.exists(abs_search_dir) and os.path.isdir(abs_search_dir):
                # Look for folders that match the market name (case-insensitive)
                for item in os.listdir(abs_search_dir):
                    item_path = os.path.join(abs_search_dir, item)
                    if os.path.isdir(item_path) and item.upper().strip() == market.upper().strip():
                        market_folder = item_path
                        logger.info(f"Found market folder: {market_folder}")
                        break
                if market_folder:
                    break
        
        # Load images from the market folder if it exists
        if market_folder:
            image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp'}
            try:
                for filename in os.listdir(market_folder):
                    file_path = os.path.join(market_folder, filename)
                    if os.path.isfile(file_path):
                        # Check if it's an image file
                        _, ext = os.path.splitext(filename.lower())
                        if ext in image_extensions:
                            # Infer label from filename (PASS or FAIL)
                            try:
                                label = infer_label(filename)
                                if label in ['PASS', 'FAIL']:
                                    # Use absolute path for the file
                                    abs_file_path = os.path.abspath(file_path)
                                    local_photos.append({
                                        'image_url': abs_file_path,  # Use file path instead of URL
                                        'evaluation_status': label,
                                        'evaluation_message': f"Loaded from local folder: {market_folder}",
                                        'market': market,
                                        'is_local_file': True  # Flag to indicate this is a local file, not URL
                                    })
                                    logger.debug(f"Added local photo: {filename} -> {label}")
                                else:
                                    logger.warning(f"Skipping image {filename}: could not infer PASS/FAIL from filename")
                            except Exception as e:
                                logger.warning(f"Error inferring label for {filename}: {e}")
                                continue
                
                logger.info(f"Loaded {len(local_photos)} local photos from folder '{market_folder}' for market '{market}'")
            except Exception as e:
                logger.warning(f"Error reading market folder {market_folder}: {e}")
        
        # Combine CSV photos and local photos
        all_photos = base_photos + local_photos
        logger.info(f"Total photos for market '{market}': {len(all_photos)} ({len(base_photos)} from CSV, {len(local_photos)} from local folder)")
        
        return all_photos, ""
        
    except Exception as e:
        logger.error(f"Error loading CSV: {type(e).__name__}: {e}", exc_info=True)
        return [], f"Error loading CSV file: {str(e)}"


# -------------------- Label + file helpers --------------------

def infer_label(filename: str) -> str:
    """Infer label from filename (PASS or FAIL). Wrapped in try-except to prevent crashes."""
    try:
        name = os.path.basename(filename).upper()
        if name.startswith("PASS"):
            return "PASS"
        if name.startswith("FAIL"):
            return "FAIL"
        raise ValueError(f"Filename must start with PASS or FAIL: {os.path.basename(filename)}")
    except Exception as e:
        logger.warning(f"Error inferring label from filename '{filename}': {e}")
        # Default to FAIL for safety (conservative approach)
        return "FAIL"

def to_data_url(filepath_or_url: str, max_size_mb: Optional[float] = None) -> str:
    """
    Convert image file or URL to data URL, compressing if necessary.
    
    Args:
        filepath_or_url: Path to image file or URL to image
        max_size_mb: Maximum size in MB for base64-encoded image (defaults to IMAGE_MAX_SIZE_MB config, typically 3.5 MB for cost savings)
    
    Returns:
        Data URL string
    """
    # Use configured max size if not provided
    if max_size_mb is None:
        max_size_mb = IMAGE_MAX_SIZE_MB if 'IMAGE_MAX_SIZE_MB' in globals() else 3.5
    # Check if it's a URL
    is_url = filepath_or_url.startswith("http://") or filepath_or_url.startswith("https://")
    
    if is_url:
        # Download image from URL
        download_start = time.time()
        logger.debug(f"Downloading image from URL: {filepath_or_url[:80]}...")
        try:
            response = requests.get(filepath_or_url, timeout=30)
            response.raise_for_status()
            original_data = response.content
            download_elapsed = time.time() - download_start
            logger.debug(f"Successfully downloaded {len(original_data)} bytes from URL in {download_elapsed:.2f}s")
            if download_elapsed > 2.0:  # Log if download takes > 2 seconds
                logger.warning(f"⚠️ Slow URL download: {filepath_or_url[:80]}... took {download_elapsed:.2f}s")
            # Try to determine mime type from content-type header or URL
            content_type = response.headers.get('content-type', '')
            if content_type and content_type.startswith('image/'):
                mime = content_type
            else:
                mime, _ = mimetypes.guess_type(filepath_or_url)
                if not mime or not mime.startswith("image/"):
                    mime = "image/jpeg"
        except Exception as e:
            logger.error(f"Error downloading image from URL {filepath_or_url}: {e}")
            raise RuntimeError(f"Failed to download image from URL: {e}")
    else:
        # Local file path
        mime, _ = mimetypes.guess_type(filepath_or_url)
        if not mime or not mime.startswith("image/"):
            mime = "image/jpeg"  # Default to JPEG
        
        # Read original file
        with open(filepath_or_url, "rb") as f:
            original_data = f.read()
    
    # Get display name for logging
    display_name = os.path.basename(filepath_or_url) if not is_url else filepath_or_url.split("/")[-1]
    
    # Check if we need to compress (5 MB = 5242880 bytes, but base64 adds ~33% overhead)
    # So we want to keep base64 size under ~4.5 MB to be safe
    max_size_bytes = int(max_size_mb * 1024 * 1024)
    
    # Estimate base64 size (base64 is ~33% larger than binary)
    estimated_base64_size = len(original_data) * 4 // 3
    
    # Quick check: if file is already small enough, skip compression entirely
    if estimated_base64_size <= max_size_bytes:
        # No compression needed - encode directly (fast path)
        b64 = base64.b64encode(original_data).decode("utf-8")
        file_size_kb = len(original_data) / 1024
        logger.debug(f"Image {display_name}: {file_size_kb:.1f}KB, no compression needed (fast path)")
    else:
        # Need to compress
        logger.info(f"Compressing image {display_name}: {len(original_data)} bytes -> target < {max_size_bytes} bytes")
        try:
            # Open and compress image
            img = Image.open(BytesIO(original_data))
            
            # Convert RGBA to RGB if necessary (removes alpha channel)
            if img.mode in ("RGBA", "LA", "P"):
                # Create white background
                background = Image.new("RGB", img.size, (255, 255, 255))
                if img.mode == "P":
                    img = img.convert("RGBA")
                background.paste(img, mask=img.split()[-1] if img.mode in ("RGBA", "LA") else None)
                img = background
            elif img.mode != "RGB":
                img = img.convert("RGB")
            
            # Calculate compression quality and size
            # Optimize: resize FIRST and more aggressively for large files to speed up processing
            file_size_mb = len(original_data) / (1024 * 1024)
            
            # More aggressive resizing for cost savings (reduces image token costs)
            if file_size_mb > 5:
                # Very large files: resize to 800px max (more aggressive)
                max_dimension = 800
            elif file_size_mb > 2:
                # Large files: resize to 1200px max (more aggressive)
                max_dimension = 1200
            else:
                # Smaller files: resize to 1600px max (more aggressive)
                max_dimension = 1600
            
            # Resize if image is too large (do this first to speed up compression)
            if max(img.size) > max_dimension:
                ratio = max_dimension / max(img.size)
                new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
                img = img.resize(new_size, Image.Resampling.LANCZOS)
                logger.debug(f"Resized image {display_name} from {img.size} to {new_size} (file was {file_size_mb:.2f}MB)")
            
            # More aggressive compression for cost savings (reduces image token costs)
            if file_size_mb > 5:
                initial_quality = 50  # Very low quality for very large files (reduced from 60)
            elif file_size_mb > 2:
                initial_quality = 60  # Low quality for large files (reduced from 70)
            else:
                initial_quality = 70  # Medium quality for smaller files (reduced from 80)
            quality = initial_quality
            
            # Try to compress to target size (reduced attempts for speed)
            for attempt in range(2):  # Only 2 attempts for speed
                output = BytesIO()
                # Use optimize=False for faster saves (slight size increase but much faster)
                img.save(output, format="JPEG", quality=quality, optimize=False)
                compressed_data = output.getvalue()
                estimated_base64_size = len(compressed_data) * 4 // 3
                
                if estimated_base64_size <= max_size_bytes:
                    b64 = base64.b64encode(compressed_data).decode("utf-8")
                    logger.info(f"Compressed image {display_name}: {len(original_data)/1024:.1f}KB -> {len(compressed_data)/1024:.1f}KB (quality={quality})")
                    break
                else:
                    # Reduce quality more aggressively for cost savings
                    quality = max(35, quality - 30)  # Larger steps, don't go below 35 (reduced from 40)
                    if attempt == 1:
                        # Last attempt - use what we have
                        b64 = base64.b64encode(compressed_data).decode("utf-8")
                        logger.warning(f"Image {display_name} still large after compression: {len(compressed_data)/1024:.1f}KB (quality={quality})")
            else:
                # Fallback: use original if compression failed
                b64 = base64.b64encode(original_data).decode("utf-8")
                logger.warning(f"Compression failed for {display_name}, using original")
                
        except Exception as e:
            # If compression fails, try original
            logger.warning(f"Error compressing image {display_name}: {e}, using original")
            b64 = base64.b64encode(original_data).decode("utf-8")
    
    return f"data:{mime};base64,{b64}"

def parse_json_strict(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    if text.startswith("{") and text.endswith("}"):
        return json.loads(text)

    # salvage first {...}
    s = text.find("{")
    e = text.rfind("}")
    if s != -1 and e != -1 and e > s:
        return json.loads(text[s:e+1])

    raise ValueError("Model response is not valid JSON")

def compute_metrics(rows: List[Dict[str, Any]], ux_focus_pct: float = 0.0) -> Dict[str, Any]:
    # PASS is positive class
    tp = sum(1 for r in rows if r["gt"] == "PASS" and r["pred"] == "PASS")
    fn = sum(1 for r in rows if r["gt"] == "PASS" and r["pred"] == "FAIL")
    fp = sum(1 for r in rows if r["gt"] == "FAIL" and r["pred"] == "PASS")
    tn = sum(1 for r in rows if r["gt"] == "FAIL" and r["pred"] == "FAIL")

    recall = (tp / (tp + fn) * 100.0) if (tp + fn) else 0.0
    precision = (tp / (tp + fp) * 100.0) if (tp + fp) else 0.0
    
    # Calculate weights (complementary)
    w_ux = ux_focus_pct / 100.0  # UX focus weight (0.0 to 1.0)
    w_gp = 1.0 - w_ux  # GP focus weight (1.0 to 0.0)
    
    # Weighted score: higher is better
    # GP focus (0%): maximize recall
    # UX focus (100%): maximize precision
    weighted_score = (w_gp * recall + w_ux * precision)
    
    # Track Logic Mismatches: cases where checklist/rule_validation contradicts the decision
    logic_mismatches = 0
    mismatch_details = []
    
    for r in rows:
        pred = r.get("pred", "")
        rule_validation = r.get("rule_validation", {})
        visual_checklist = r.get("visual_checklist", {})
        
        # Check for contradictions
        is_blocking = rule_validation.get("is_blocking_path", False)
        is_prohibited = rule_validation.get("is_in_prohibited_zone", False)
        is_upright = rule_validation.get("is_upright", True)
        posture = visual_checklist.get("posture", "").lower() if isinstance(visual_checklist.get("posture"), str) else ""
        
        has_mismatch = False
        mismatch_reasons = []
        
        if pred == "PASS":
            # PASS decision should not have blocking/prohibited violations
            if is_blocking:
                has_mismatch = True
                mismatch_reasons.append("blocking_path=True")
            if is_prohibited:
                has_mismatch = True
                mismatch_reasons.append("in_prohibited_zone=True")
            if not is_upright:
                has_mismatch = True
                mismatch_reasons.append("is_upright=False")
            if posture == "tipped":
                has_mismatch = True
                mismatch_reasons.append("posture=tipped")
        elif pred == "FAIL":
            # FAIL decision should have at least one violation
            # If all checks pass, it might be a mismatch (but less certain)
            if not is_blocking and not is_prohibited and is_upright and posture in ("upright", ""):
                # This is a potential mismatch - decision is FAIL but no clear violations
                has_mismatch = True
                mismatch_reasons.append("no_violations_detected")
        
        if has_mismatch:
            logic_mismatches += 1
            mismatch_details.append({
                "file": r.get("file", "unknown"),
                "pred": pred,
                "reasons": ", ".join(mismatch_reasons),
                "rule_validation": rule_validation,
                "visual_checklist": visual_checklist
            })
    
    total_with_checklist = sum(1 for r in rows if r.get("rule_validation") or r.get("visual_checklist"))
    logic_mismatch_pct = (logic_mismatches / total_with_checklist * 100.0) if total_with_checklist > 0 else 0.0
    
    return {
        "tp": tp, "fn": fn, "fp": fp, "tn": tn,
        "recall_pass_pct": round(recall, 2),
        "precision_pass_pct": round(precision, 2),
        "weighted_score": round(weighted_score, 2),
        "ux_focus_pct": round(ux_focus_pct, 1),
        "logic_mismatches": logic_mismatches,
        "logic_mismatch_pct": round(logic_mismatch_pct, 2),
        "logic_mismatch_details": mismatch_details,
    }


# -------------------- OpenRouter calls --------------------

# Global cache for API responses (keyed by deterministic hash)
_api_cache: Dict[str, Dict[str, Any]] = {}
_cache_lock = threading.Lock()

# Global cost tracking
_cost_tracker: Dict[str, float] = {
    "vision_calls": 0.0,  # Total cost for vision model calls
    "optimizer_calls": 0.0,  # Total cost for optimizer calls
    "total": 0.0
}
_cost_lock = threading.Lock()

# Model pricing (per 1M tokens) - update these based on current OpenRouter pricing
MODEL_PRICING = {
    "anthropic/claude-sonnet-4": {"input": 3.0, "output": 15.0},  # $3/$15 per 1M tokens
    "claude-sonnet-4": {"input": 3.0, "output": 15.0},
    "amazon/nova-2-lite-v1": {"input": 0.1, "output": 0.4},  # Placeholder pricing - update with actual pricing
    "nova-2-lite-v1": {"input": 0.1, "output": 0.4},  # Placeholder pricing - update with actual pricing
    "openai/gpt-4o-mini": {"input": 0.15, "output": 0.6},  # $0.15/$0.6 per 1M tokens
    "anthropic/claude-3-haiku": {"input": 0.25, "output": 1.25},  # $0.25/$1.25 per 1M tokens
}

def _generate_grounded_correct_logic(error_row: Dict[str, Any], error_type: str, item: Optional["ImageItem"] = None) -> str:
    """
    Generate grounded correct_logic with observable visual facts instead of generic text.
    This helps the model learn better discrimination.
    """
    wrong_reason = error_row.get('reason', '')
    rule_validation = error_row.get('rule_validation', {})
    visual_checklist = error_row.get('visual_checklist', {})
    
    # Extract observable facts
    is_blocking = rule_validation.get('is_blocking_path', False)
    is_prohibited = rule_validation.get('is_in_prohibited_zone', False)
    is_upright = rule_validation.get('is_upright', True)
    posture = visual_checklist.get('posture', '').lower() if isinstance(visual_checklist.get('posture'), str) else ''
    infrastructure = visual_checklist.get('infrastructure_proximity', '')
    
    # Build grounded explanation with observable facts
    facts = []
    if infrastructure and infrastructure.strip() and infrastructure.lower() not in ['none', 'no', '']:
        facts.append(f"infrastructure: {infrastructure}")
    if posture in ['upright', 'tipped', 'leaning']:
        facts.append(f"posture: {posture}")
    if is_blocking is not None:
        facts.append(f"blocking_path: {is_blocking}")
    if is_prohibited is not None:
        facts.append(f"prohibited_zone: {is_prohibited}")
    if is_upright is not None:
        facts.append(f"upright: {is_upright}")
    
    facts_text = "; ".join(facts) if facts else "no violations detected"
    
    if error_type == 'false_negative':
        return f"This is a valid parking case (PASS) that was incorrectly rejected. The model said: '{wrong_reason}'. This is wrong because observable facts show: {facts_text}. The vehicle meets all parking requirements."
    else:  # false_positive
        return f"This is an invalid parking case (FAIL) that was incorrectly accepted. The model said: '{wrong_reason}'. This is wrong because observable facts show: {facts_text}. The parking violates one or more rules."

def _compute_decision_from_rules(rule_validation: Dict[str, Any], visual_checklist: Dict[str, Any], model_decision: str) -> str:
    """
    Compute deterministic PASS/FAIL decision from rule_validation and visual_checklist.
    This reduces variance and makes optimization more stable.
    
    CONSERVATIVE APPROACH: Only override model decision if rule_validation is complete and confident.
    If rule_validation is incomplete or ambiguous, trust the model's decision.
    """
    # Check if rule_validation is complete (all fields present and not None)
    is_blocking = rule_validation.get("is_blocking_path")
    is_prohibited = rule_validation.get("is_in_prohibited_zone")
    is_upright = rule_validation.get("is_upright")
    
    # If rule_validation is incomplete, trust model decision
    if is_blocking is None and is_prohibited is None and is_upright is None:
        return model_decision  # Fallback to model if no rule_validation data
    
    # Extract visual checklist
    posture = visual_checklist.get("posture", "").lower() if isinstance(visual_checklist.get("posture"), str) else ""
    
    # Use defaults for None values (conservative - don't assume violations)
    is_blocking = is_blocking if is_blocking is not None else False
    is_prohibited = is_prohibited if is_prohibited is not None else False
    is_upright = is_upright if is_upright is not None else True
    
    # Deterministic rules: FAIL if CLEAR violation detected
    # Only override model if we have high confidence in the violation
    if (is_blocking is True) or (is_prohibited is True) or (is_upright is False) or (posture == "tipped"):
        return "FAIL"
    
    # If no clear violations detected, use model decision (more nuanced than simple PASS)
    # This allows model to consider edge cases that deterministic rules might miss
    return model_decision

def _calculate_cost(usage: Dict[str, Any], model_id: str) -> float:
    """Calculate cost from OpenRouter usage data."""
    if not usage:
        return 0.0
    
    # Get pricing for model (try exact match first, then fallback)
    pricing = MODEL_PRICING.get(model_id) or MODEL_PRICING.get(model_id.split("/")[-1])
    if not pricing:
        logger.warning(f"Unknown pricing for model {model_id}, using Claude Sonnet 4 pricing")
        pricing = MODEL_PRICING["anthropic/claude-sonnet-4"]
    
    # Extract token counts
    prompt_tokens = usage.get("prompt_tokens", 0)
    completion_tokens = usage.get("completion_tokens", 0)
    
    # Calculate cost (pricing is per 1M tokens)
    input_cost = (prompt_tokens / 1_000_000) * pricing["input"]
    output_cost = (completion_tokens / 1_000_000) * pricing["output"]
    
    # For vision models, also account for image tokens
    # OpenRouter typically includes this in prompt_tokens, but we can check
    total_cost = input_cost + output_cost
    
    return total_cost

def _track_cost(cost: float, call_type: str = "vision_calls") -> None:
    """Track cost for a specific call type."""
    with _cost_lock:
        _cost_tracker[call_type] += cost
        _cost_tracker["total"] += cost

def get_cost_summary() -> Dict[str, Any]:
    """Get current cost summary."""
    with _cost_lock:
        return _cost_tracker.copy()

def reset_cost_tracker() -> None:
    """Reset cost tracker."""
    with _cost_lock:
        _cost_tracker["vision_calls"] = 0.0
        _cost_tracker["optimizer_calls"] = 0.0
        _cost_tracker["total"] = 0.0

def _load_persistent_cache() -> None:
    """Load cache from disk if persistent caching is enabled."""
    if not (PERSISTENT_CACHE_ENABLED if 'PERSISTENT_CACHE_ENABLED' in globals() else False):
        return
    
    cache_dir = CACHE_DIR if 'CACHE_DIR' in globals() else "./.api_cache"
    cache_file = os.path.join(cache_dir, "api_cache.json")
    
    if os.path.exists(cache_file):
        try:
            with open(cache_file, 'r') as f:
                loaded_cache = json.load(f)
                with _cache_lock:
                    _api_cache.update(loaded_cache)
                logger.info(f"✅ Loaded {len(loaded_cache)} cached responses from disk")
        except Exception as e:
            logger.warning(f"Failed to load persistent cache: {e}")

def _save_persistent_cache() -> None:
    """Save cache to disk if persistent caching is enabled."""
    if not (PERSISTENT_CACHE_ENABLED if 'PERSISTENT_CACHE_ENABLED' in globals() else False):
        return
    
    cache_dir = CACHE_DIR if 'CACHE_DIR' in globals() else "./.api_cache"
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, "api_cache.json")
    
    try:
        with _cache_lock:
            cache_copy = _api_cache.copy()
        
        with open(cache_file, 'w') as f:
            json.dump(cache_copy, f)
        logger.debug(f"Saved {len(cache_copy)} cached responses to disk")
    except Exception as e:
        logger.warning(f"Failed to save persistent cache: {e}")

# Load cache on startup
_load_persistent_cache()

def get_cache_stats() -> Dict[str, Any]:
    """Get cache statistics."""
    with _cache_lock:
        return {
            "cache_size": len(_api_cache),
            "cache_keys": list(_api_cache.keys())[:10]  # First 10 keys for debugging
        }

def clear_cache() -> None:
    """Clear the API response cache (both memory and disk)."""
    with _cache_lock:
        _api_cache.clear()
        logger.info("API cache cleared from memory")
    
    # Also clear disk cache
    cache_dir = CACHE_DIR if 'CACHE_DIR' in globals() else "./.api_cache"
    cache_file = os.path.join(cache_dir, "api_cache.json")
    if os.path.exists(cache_file):
        try:
            os.remove(cache_file)
            logger.info("Persistent cache file deleted")
        except Exception as e:
            logger.warning(f"Failed to delete persistent cache file: {e}")

def _generate_cache_key(
    system_prompt: str,
    user_prompt: str,
    image_data_url: str,
    reference_examples: Optional[List[Dict[str, Any]]],
    model_id: str,
    temperature: float,
    json_contract: str = "",
    **kwargs
) -> str:
    """Generate deterministic cache key from all relevant parameters."""
    # Hash the image (extract base64 data from data URL)
    image_hash = ""
    if image_data_url.startswith("data:"):
        # Extract base64 part
        try:
            base64_data = image_data_url.split(",", 1)[1] if "," in image_data_url else ""
            image_bytes = base64.b64decode(base64_data)
            image_hash = hashlib.sha256(image_bytes).hexdigest()
        except Exception as e:
            logger.warning(f"Error hashing image: {e}, using full URL hash")
            image_hash = hashlib.sha256(image_data_url.encode()).hexdigest()
    else:
        image_hash = hashlib.sha256(image_data_url.encode()).hexdigest()
    
    # Hash prompts (include json_contract in user_prompt hash since it's appended)
    system_prompt_hash = hashlib.sha256(system_prompt.encode()).hexdigest()
    full_user_prompt = (user_prompt or "").strip() + "\n\n" + (json_contract or "")
    user_prompt_hash = hashlib.sha256(full_user_prompt.encode()).hexdigest()
    
    # Hash reference examples (if any) - CRITICAL: must include image hashes
    ref_examples_hash = ""
    if reference_examples:
        # Create a deterministic representation of reference examples INCLUDING image hashes
        ref_examples_repr = []
        for ex in reference_examples:
            # Hash the reference example image (CRITICAL FIX: images affect few-shot context)
            ref_image_hash = ""
            ref_data_url = ex.get("data_url", "")
            if ref_data_url and ref_data_url.startswith("data:"):
                try:
                    base64_data = ref_data_url.split(",", 1)[1] if "," in ref_data_url else ""
                    image_bytes = base64.b64decode(base64_data)
                    ref_image_hash = hashlib.sha256(image_bytes).hexdigest()
                except Exception as e:
                    logger.warning(f"Error hashing reference example image: {e}")
                    ref_image_hash = hashlib.sha256(ref_data_url.encode()).hexdigest()
            elif ref_data_url:
                ref_image_hash = hashlib.sha256(ref_data_url.encode()).hexdigest()
            
            ref_examples_repr.append({
                "type": ex.get("type", ""),
                "gt": ex.get("gt", ""),
                "pred": ex.get("pred", ""),
                "wrong_reason": ex.get("wrong_reason", ""),
                "correct_logic": ex.get("correct_logic", ""),
                "image_hash": ref_image_hash,  # CRITICAL: include image hash
            })
        ref_examples_str = json.dumps(ref_examples_repr, sort_keys=True)
        ref_examples_hash = hashlib.sha256(ref_examples_str.encode()).hexdigest()
    
    # Combine all components
    cache_components = {
        "image_hash": image_hash,
        "model_id": model_id,
        "system_prompt_hash": system_prompt_hash,
        "user_prompt_hash": user_prompt_hash,
        "ref_examples_hash": ref_examples_hash,
        "temperature": temperature,
        # Add other decoding params if needed (top_p, max_tokens, etc.)
    }
    
    # Create deterministic JSON string and hash it
    cache_key_str = json.dumps(cache_components, sort_keys=True)
    cache_key = hashlib.sha256(cache_key_str.encode()).hexdigest()
    
    return cache_key

def openrouter_classify(system_prompt: str, user_prompt: str, image_data_url: str, reference_examples: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    logger.debug("Starting openrouter_classify")
    if not OPENROUTER_API_KEY:
        error_msg = "Missing OPENROUTER_API_KEY environment variable"
        logger.error(error_msg)
        raise RuntimeError(error_msg)
    
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:7860",
        "X-Title": "Gradio Prompt Optimizer (No Projects)",
    }

    json_contract = """
Return ONLY valid JSON. 
Schema:
{
  "visual_checklist": {
    "posture": "upright | tipped | leaning",
    "visibility": "fully_visible | partially_occluded",
    "infrastructure_proximity": "list any nearby ramps, doors, or crosswalks",
    "has_clear_path": boolean
  },
  "rule_validation": {
    "is_blocking_path": boolean,
    "is_in_prohibited_zone": boolean,
    "is_upright": boolean
  },
  "decision": "PASS" | "FAIL",
  "reason": "short explanation (max 20 words)"
}
""".strip()
    
    # Generate cache key (must be done before building messages)
    cache_key = _generate_cache_key(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        image_data_url=image_data_url,
        reference_examples=reference_examples,
        model_id=MODEL_ID,
        temperature=MODEL_TEMPERATURE,
        json_contract=json_contract
    )
    
    # Check cache first
    with _cache_lock:
        if cache_key in _api_cache:
            logger.debug(f"Cache HIT for image classification (key: {cache_key[:16]}...)")
            cached_result = _api_cache[cache_key].copy()
            logger.info("✅ Using cached result - $0 API cost")
            return cached_result
    
    logger.debug(f"Cache MISS - making API call (key: {cache_key[:16]}...)")
    
    # Optional: Use cheaper model if metrics are very good (cost optimization #6)
    model_to_use = MODEL_ID
    if 'CHEAPER_VISION_MODEL' in globals() and 'USE_CHEAPER_MODEL_THRESHOLD' in globals():
        # This would require passing previous metrics - for now, we'll use the main model
        # Can be enhanced later to check metrics from previous iteration
        pass

    # Build messages array with reference examples
    # Add cache_control to static content blocks for OpenRouter Prompt Caching
    cache_control = {"provider": {"cache_control": {"type": "ephemeral"}}}
    
    messages = [
        {
            "role": "system",
            "content": (system_prompt or "").strip(),
            **cache_control
        },
    ]
    
    # Add reference examples as few-shot learning examples
    # These are static across the batch, so add cache_control
    if reference_examples:
        for example in reference_examples:
            example_text = f"Reference Example ({'False Negative' if example['type'] == 'false_negative' else 'False Positive'}):\n"
            example_text += f"Ground Truth: {example['gt']}\n"
            example_text += f"Previous Model Prediction: {example['pred']}\n"
            example_text += f"Previous Model Reason: {example['wrong_reason']}\n"
            example_text += f"Correct Logic: {example['correct_logic']}\n"
            example_text += f"Correct Decision: {example['gt']}"
            
            messages.append({
                "role": "user",
                "content": [
                    {"type": "text", "text": example_text, **cache_control},
                    {"type": "image_url", "image_url": {"url": example['data_url']}, **cache_control},
                ]
            })
            # Add assistant response showing the correct decision with visual checklist structure
            # Note: For reference examples, we provide a minimal structure since we don't have the actual measurements
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "visual_checklist": {
                        "posture": "upright",
                        "visibility": "fully_visible",
                        "infrastructure_proximity": "",
                        "has_clear_path": False
                    },
                    "rule_validation": {
                        "is_blocking_path": example['gt'] == 'FAIL',
                        "is_in_prohibited_zone": example['gt'] == 'FAIL',
                        "is_upright": True
                    },
                    "decision": example['gt'],
                    "reason": example['correct_logic']
                }),
                **cache_control
            })
    
    # Add the actual evaluation request
    # Cache the user_prompt rules (static), but NOT the image (dynamic per request)
    messages.append({
        "role": "user",
        "content": [
            {"type": "text", "text": (user_prompt or "").strip() + "\n\n" + json_contract, **cache_control},
            {"type": "image_url", "image_url": {"url": image_data_url}},  # Image is dynamic, no cache
        ]
    })
    
    payload = {
        "model": MODEL_ID,
        "temperature": MODEL_TEMPERATURE,
        "messages": messages,
    }

    try:
        api_call_start = time.time()
        logger.debug(f"Calling OpenRouter API with model {MODEL_ID}")
        data = post_with_retries(url, headers, payload)
        api_call_elapsed = time.time() - api_call_start
        logger.debug(f"Received response from OpenRouter in {api_call_elapsed:.2f}s")
        if api_call_elapsed > 10.0:  # Log if API call takes > 10 seconds
            logger.warning(f"⚠️ Slow API call: took {api_call_elapsed:.2f}s")
        
        # Extract and track cost with enhanced logging
        usage = data.get("usage", {})
        cache_hit = False
        with _cache_lock:
            cache_hit = cache_key in _api_cache
        
        if usage:
            cost = _calculate_cost(usage, MODEL_ID)
            _track_cost(cost, "vision_calls")
            # Enhanced logging: model, tokens, image size estimate, cache status
            image_size_kb = len(image_data_url) / 1024 if image_data_url else 0
            logger.info(f"Vision call: model={MODEL_ID}, tokens={usage.get('prompt_tokens', 0)}/{usage.get('completion_tokens', 0)}, image_size={image_size_kb:.1f}KB, cache_hit={cache_hit}, cost=${cost:.4f} (ESTIMATE)")
        
        # Check for errors in response
        if "error" in data:
            error_info = data.get("error", {})
            error_msg = error_info.get("message", "Unknown error")
            # Try to extract more details from metadata
            if "metadata" in error_info and "raw" in error_info["metadata"]:
                try:
                    raw_error = json.loads(error_info["metadata"]["raw"])
                    if "error" in raw_error and "message" in raw_error["error"]:
                        error_msg = raw_error["error"]["message"]
                except:
                    pass
            raise RuntimeError(f"API returned error: {error_msg}")
        
        if "choices" not in data or len(data["choices"]) == 0:
            error_msg = f"Invalid response structure: {data}"
            logger.error(error_msg)
            raise ValueError(error_msg)
        
        content = data["choices"][0]["message"]["content"]
        logger.debug(f"Response content length: {len(content)}")
        
        parsed = parse_json_strict(content)
        logger.debug(f"Parsed JSON successfully: {parsed}")

        decision = str(parsed.get("decision", "")).upper().strip()
        if decision not in ("PASS", "FAIL"):
            error_msg = f"Invalid decision: {decision}"
            logger.error(error_msg)
            raise ValueError(error_msg)

        visual_checklist = parsed.get("visual_checklist", {})
        rule_validation = parsed.get("rule_validation", {})
        
        # Optionally use deterministic decision from rule_validation (disabled by default)
        # When enabled, this reduces variance but may be too strict
        use_deterministic = USE_DETERMINISTIC_DECISION if 'USE_DETERMINISTIC_DECISION' in globals() else False
        if use_deterministic:
            deterministic_decision = _compute_decision_from_rules(rule_validation, visual_checklist, decision)
            final_decision = deterministic_decision
        else:
            final_decision = decision  # Use model's decision (more flexible)
            deterministic_decision = None
        
        result = {
            "decision": final_decision,  # Use model decision (or deterministic if enabled)
            "model_decision": decision,  # Keep original for comparison
            "deterministic_decision": deterministic_decision,  # For analysis
            "reason": parsed.get("reason", ""),
            "visual_checklist": visual_checklist,
            "rule_validation": rule_validation
        }
        if use_deterministic:
            logger.debug(f"Classification result: {result} (model: {decision}, deterministic: {deterministic_decision}, using: {final_decision})")
        else:
            logger.debug(f"Classification result: {result} (using model decision: {final_decision})")
        
        # Store in cache for future use
        with _cache_lock:
            _api_cache[cache_key] = result.copy()
            logger.debug(f"Cached result (cache size: {len(_api_cache)} entries)")
        
        # Save to persistent cache (async save to avoid blocking)
        # Save every 10 cache entries to reduce disk I/O
        if len(_api_cache) % 10 == 0:
            _save_persistent_cache()
        
        return result
    except Exception as e:
        logger.error(f"Error in openrouter_classify: {type(e).__name__}: {str(e)}")
        raise

def analyze_error_patterns(errors: List[Dict[str, Any]], error_type: str = "errors") -> str:
    """
    Analyze error patterns from the reasons provided by the model.
    Returns a summary of common patterns found in the error reasons.
    """
    if not errors:
        return f"No {error_type} to analyze."
    
    reasons = [e.get('reason', '').lower() for e in errors if e.get('reason')]
    if not reasons:
        return f"Found {len(errors)} {error_type}, but no reasons available for pattern analysis."
    
    # Common keywords/phrases to look for
    keywords = {
        'crosswalk': ['crosswalk', 'crossing', 'pedestrian crossing'],
        'sidewalk': ['sidewalk', 'walkway', 'pedestrian path'],
        'blocking': ['blocking', 'blocked', 'obstruct', 'obstruction'],
        'driveway': ['driveway', 'drive way'],
        'entrance': ['entrance', 'entry', 'doorway'],
        'designated': ['designated', 'parking space', 'parking zone', 'marked'],
        'furniture zone': ['furniture zone', 'furniture'],
        'upright': ['upright', 'tipped', 'fallen', 'fallen over'],
        'visible': ['visible', 'cut off', 'partial', 'incomplete'],
        'ramp': ['ramp', 'handicap', 'accessible', 'ada'],
    }
    
    # Count keyword occurrences
    pattern_counts = {}
    for keyword, variations in keywords.items():
        count = sum(1 for reason in reasons if any(var in reason for var in variations))
        if count > 0:
            pattern_counts[keyword] = count
    
    # Build summary
    if not pattern_counts:
        return f"Found {len(errors)} {error_type}, but no clear patterns detected in reasons."
    
    # Sort by frequency
    sorted_patterns = sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True)
    
    # Create summary
    total = len(errors)
    pattern_lines = []
    for pattern, count in sorted_patterns[:5]:  # Top 5 patterns
        pct = (count / total) * 100
        pattern_lines.append(f"- {pattern.replace('_', ' ').title()}: {count}/{total} ({pct:.0f}%)")
    
    summary = f"Error Pattern Analysis ({len(errors)} {error_type}):\n" + "\n".join(pattern_lines)
    return summary


def propose_next_user_prompt(
    system_prompt: str,
    current_user_prompt: str,
    false_negatives: List[Dict[str, Any]],
    false_positives: List[Dict[str, Any]],
    current_metrics: Dict[str, Any],
    items: List["ImageItem"],
    ux_focus_pct: float = 0.0,
) -> Tuple[str, List[Dict[str, Any]]]:
    logger.info(f"Proposing next user prompt (FN: {len(false_negatives)}, FP: {len(false_positives)})")
    if not OPENROUTER_API_KEY:
        error_msg = "Missing OPENROUTER_API_KEY environment variable"
        logger.error(error_msg)
        raise RuntimeError(error_msg)

    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:7860",
        "X-Title": "Gradio Prompt Optimizer (No Projects)",
    }

    # Conditionally create reference examples based on metrics (cost optimization)
    # Only use reference examples when metrics are below threshold (model needs help)
    reference_examples = []
    
    # Check if we should use reference examples
    should_use_ref_examples = True  # Default to True for first iteration (no metrics yet)
    if current_metrics:
        recall = current_metrics.get('recall_pass_pct', 0)
        precision = current_metrics.get('precision_pass_pct', 0)
        threshold = REFERENCE_EXAMPLES_THRESHOLD if 'REFERENCE_EXAMPLES_THRESHOLD' in globals() else 90.0
        should_use_ref_examples = (recall < threshold) or (precision < threshold)
        if not should_use_ref_examples:
            logger.info(f"Metrics above threshold (recall: {recall}%, precision: {precision}%) - skipping reference examples to save costs")
    
    if should_use_ref_examples:
        logger.info(f"Metrics below threshold - creating reference examples for few-shot learning")
        items_by_filename = {os.path.basename(item.file): item for item in items}
        
        # Select 2 most clear False Negatives (GT PASS, Pred FAIL)
        selected_fn = false_negatives[:2] if len(false_negatives) >= 2 else false_negatives
        for fn in selected_fn:
            filename = fn['file']
            item = items_by_filename.get(filename)
            if item:
                try:
                    data_url = item.get_data_url()
                    reference_examples.append({
                        'type': 'false_negative',
                        'gt': 'PASS',
                        'pred': 'FAIL',
                        'data_url': data_url,
                        'filename': filename,
                    'wrong_reason': fn.get('reason', ''),
                    'correct_logic': _generate_grounded_correct_logic(fn, 'false_negative', items_by_filename.get(filename))
                    })
                except Exception as e:
                    logger.warning(f"Could not get data URL for reference example {filename}: {e}")
        
        # Select 2 most clear False Positives (GT FAIL, Pred PASS)
        selected_fp = false_positives[:2] if len(false_positives) >= 2 else false_positives
        for fp in selected_fp:
            filename = fp['file']
            item = items_by_filename.get(filename)
            if item:
                try:
                    data_url = item.get_data_url()
                    reference_examples.append({
                        'type': 'false_positive',
                        'gt': 'FAIL',
                        'pred': 'PASS',
                        'data_url': data_url,
                        'filename': filename,
                    'wrong_reason': fp.get('reason', ''),
                    'correct_logic': _generate_grounded_correct_logic(fp, 'false_positive', items_by_filename.get(filename))
                    })
                except Exception as e:
                    logger.warning(f"Could not get data URL for reference example {filename}: {e}")
    else:
        logger.info("Skipping reference examples creation (metrics above threshold)")

    fn_lines = "\n".join([
        f"- {x['file']}: predicted FAIL but should PASS. Model reason: {x.get('reason','')}"
        for x in false_negatives[:20]
    ]) or "- None"

    fp_lines = "\n".join([
        f"- {x['file']}: predicted PASS but should FAIL. Model reason: {x.get('reason','')}"
        for x in false_positives[:20]
    ]) or "- None"
    
    # Create reference examples text for the optimizer prompt
    # IMPORTANT: Do NOT include filenames - only general patterns
    reference_examples_text = ""
    if reference_examples:
        reference_examples_text = "\n\nMOST CLEAR REFERENCE EXAMPLES (to be included in ### REFERENCE EXAMPLES section):\n"
        reference_examples_text += "NOTE: These are GENERAL patterns. DO NOT include filenames, URLs, or specific image identifiers in your output.\n"
        for idx, example in enumerate(reference_examples, 1):
            example_type = "False Negative" if example['type'] == 'false_negative' else "False Positive"
            reference_examples_text += f"\nExample {idx} ({example_type} Pattern):\n"
            # DO NOT include filename - only general pattern description
            reference_examples_text += f"- Ground Truth: {example['gt']}\n"
            reference_examples_text += f"- Previous Model Prediction: {example['pred']}\n"
            reference_examples_text += f"- Previous Model Reason: {example['wrong_reason']}\n"
            reference_examples_text += f"- Correct Logic: {example['correct_logic']}\n"

    w_ux = ux_focus_pct / 100.0
    w_gp = 1.0 - w_ux
    focus_description = f"""
Optimization Focus: {ux_focus_pct}% UX / {100-ux_focus_pct}% GP
- GP Focus ({100-ux_focus_pct}%): Maximize recall (catch valid parking), minimize false negatives
- UX Focus ({ux_focus_pct}%): Maximize precision (avoid false alarms), minimize false positives
Weighted Score = {w_gp:.1%} × Recall + {w_ux:.1%} × Precision
Maximize this weighted score.
"""
    
    # Analyze error patterns
    fn_patterns = analyze_error_patterns(false_negatives, "false negatives")
    fp_patterns = analyze_error_patterns(false_positives, "false positives")
    
    # Get current metrics
    current_recall = current_metrics.get('recall_pass_pct', 0)
    current_precision = current_metrics.get('precision_pass_pct', 0)
    current_weighted = current_metrics.get('weighted_score', 0)
    
    # Analyze logic mismatches to determine if model sees features but makes wrong decisions vs missing features
    logic_mismatch_details = current_metrics.get('logic_mismatch_details', [])
    logic_mismatch_analysis = ""
    
    if logic_mismatch_details:
        seeing_but_wrong = []  # Model sees features correctly but makes wrong decision
        missing_features = []  # Model misses features entirely
        
        for mismatch in logic_mismatch_details:
            visual_checklist = mismatch.get('visual_checklist', {})
            rule_validation = mismatch.get('rule_validation', {})
            pred = mismatch.get('pred', '')
            infrastructure = visual_checklist.get('infrastructure_proximity', '').lower()
            posture = visual_checklist.get('posture', '').lower()
            visibility = visual_checklist.get('visibility', '').lower()
            is_blocking = rule_validation.get('is_blocking_path', False)
            is_prohibited = rule_validation.get('is_in_prohibited_zone', False)
            is_upright = rule_validation.get('is_upright', True)
            
            # Check if model is seeing features
            has_infrastructure_info = bool(infrastructure and infrastructure.strip() and infrastructure not in ['', 'none', 'no'])
            has_posture_info = bool(posture and posture in ['upright', 'tipped', 'leaning'])
            has_visibility_info = bool(visibility and visibility in ['fully_visible', 'partially_occluded'])
            has_rule_info = is_blocking is not None or is_prohibited is not None or is_upright is not None
            
            # Determine if model is seeing features but making wrong decision
            if has_infrastructure_info or has_posture_info or has_visibility_info or has_rule_info:
                # Model is seeing features - check if decision contradicts what it sees
                if pred == "PASS" and (is_blocking or is_prohibited or not is_upright or posture == "tipped"):
                    seeing_but_wrong.append({
                        "file": mismatch.get('file', 'unknown'),
                        "issue": "Model sees violations (blocking/prohibited/tipped) but still decides PASS",
                        "visual_checklist": visual_checklist,
                        "rule_validation": rule_validation
                    })
                elif pred == "FAIL" and not is_blocking and not is_prohibited and is_upright and posture in ("upright", ""):
                    seeing_but_wrong.append({
                        "file": mismatch.get('file', 'unknown'),
                        "issue": "Model sees no violations but still decides FAIL",
                        "visual_checklist": visual_checklist,
                        "rule_validation": rule_validation
                    })
            else:
                # Model is missing features - check what should have been detected
                missing_features.append({
                    "file": mismatch.get('file', 'unknown'),
                    "issue": "Model missing visual feature detection",
                    "visual_checklist": visual_checklist,
                    "rule_validation": rule_validation
                })
        
        # Build analysis text
        if seeing_but_wrong or missing_features:
            logic_mismatch_analysis = "\n\nLOGIC MISMATCH ANALYSIS:\n"
            logic_mismatch_analysis += f"Found {len(logic_mismatch_details)} cases where the model's decision contradicts its checklist.\n\n"
            
            if seeing_but_wrong:
                logic_mismatch_analysis += f"⚠️ SEEING FEATURES BUT WRONG DECISION ({len(seeing_but_wrong)} cases):\n"
                logic_mismatch_analysis += "The model is correctly detecting visual features (ramps, posture, blocking, etc.) but making incorrect decisions.\n"
                logic_mismatch_analysis += "ACTION REQUIRED: Sharpen the Parking Rules to better connect detected features to PASS/FAIL decisions.\n"
                logic_mismatch_analysis += "Examples:\n"
                for case in seeing_but_wrong[:5]:  # Show up to 5 examples
                    logic_mismatch_analysis += f"- {case['file']}: {case['issue']}\n"
                    if case['visual_checklist'].get('infrastructure_proximity'):
                        logic_mismatch_analysis += f"  Infrastructure: {case['visual_checklist'].get('infrastructure_proximity')}\n"
                    logic_mismatch_analysis += f"  Rule validation: blocking={case['rule_validation'].get('is_blocking_path')}, prohibited={case['rule_validation'].get('is_in_prohibited_zone')}, upright={case['rule_validation'].get('is_upright')}\n"
                logic_mismatch_analysis += "\n"
            
            if missing_features:
                logic_mismatch_analysis += f"⚠️ MISSING FEATURES ({len(missing_features)} cases):\n"
                logic_mismatch_analysis += "The model is not detecting important visual features (ramps, crosswalks, posture issues, etc.).\n"
                logic_mismatch_analysis += "ACTION REQUIRED: Add Visual Scrutiny Instructions to the prompt to guide the model to systematically check for:\n"
                logic_mismatch_analysis += "- Nearby infrastructure (ramps, doors, crosswalks)\n"
                logic_mismatch_analysis += "- Vehicle posture (upright vs tipped vs leaning)\n"
                logic_mismatch_analysis += "- Path blocking assessment\n"
                logic_mismatch_analysis += "- Prohibited zone detection\n"
                logic_mismatch_analysis += "Examples:\n"
                for case in missing_features[:5]:  # Show up to 5 examples
                    logic_mismatch_analysis += f"- {case['file']}: {case['issue']}\n"
                logic_mismatch_analysis += "\n"
            
            logic_mismatch_analysis += "Based on this analysis:\n"
            if seeing_but_wrong:
                logic_mismatch_analysis += "1. SHARPEN PARKING RULES: Make the connection between detected features and decisions more explicit.\n"
            if missing_features:
                logic_mismatch_analysis += "2. ADD VISUAL SCRUTINY INSTRUCTIONS: Add a systematic checklist for the model to follow when analyzing images.\n"
    
    # Determine focus type
    focus_type = "false negatives (recall)" if w_gp > w_ux else "false positives (precision)" if w_ux > w_gp else "both (balanced)"
    
    # Calculate gap to target (default to 95% if not defined)
    try:
        target_recall = TARGET_RECALL
        target_precision = TARGET_PRECISION
    except NameError:
        target_recall = 95.0
        target_precision = 95.0
    
    recall_gap = max(0, target_recall - current_recall)
    precision_gap = max(0, target_precision - current_precision)
    
    # Determine which needs more improvement
    if recall_gap > precision_gap:
        primary_focus = f"Recall needs {recall_gap:.1f}% improvement (currently {current_recall}%, target {target_recall}%)"
        secondary_focus = f"Precision needs {precision_gap:.1f}% improvement (currently {current_precision}%, target {target_precision}%)"
    else:
        primary_focus = f"Precision needs {precision_gap:.1f}% improvement (currently {current_precision}%, target {target_precision}%)"
        secondary_focus = f"Recall needs {recall_gap:.1f}% improvement (currently {current_recall}%, target {target_recall}%)"
    
    prompt = f"""
You are an expert prompt engineer specializing in image classification tasks.

PRIMARY GOAL: Achieve {target_recall}% recall AND {target_precision}% precision simultaneously.

Current Performance:
- Recall: {current_recall}% (Target: {target_recall}%, Gap: {recall_gap:.1f}%)
- Precision: {current_precision}% (Target: {target_precision}%, Gap: {precision_gap:.1f}%)
- Weighted Score: {current_weighted}%

Optimization Strategy:
{focus_description}

Priority Focus:
1. {primary_focus}
2. {secondary_focus}

You MUST output ONLY the revised USER PROMPT text (no commentary, no markdown).

CRITICAL REQUIREMENTS:
1. You MUST output ONLY the revised USER PROMPT text (no commentary, no markdown, no explanations).
2. Do not change the system prompt structure.
3. Make specific, actionable improvements based on the error patterns below.
4. Be precise and clear - avoid vague language.
5. If false negatives are the issue, make the prompt MORE lenient for valid parking scenarios.
6. If false positives are the issue, make the prompt MORE strict about what constitutes valid parking.
7. Balance both concerns - don't optimize for one metric at the expense of the other.

Current USER PROMPT:
{current_user_prompt}

ERROR ANALYSIS - False Negatives (GT PASS, predicted FAIL) - {len(false_negatives)} total:
These are valid parking cases that were incorrectly rejected. The model is being too strict.

Pattern Analysis:
{fn_patterns}

Specific Examples ({min(len(false_negatives), 20)} of {len(false_negatives)}):
{fn_lines}

ERROR ANALYSIS - False Positives (GT FAIL, predicted PASS) - {len(false_positives)} total:
These are invalid parking cases that were incorrectly accepted. The model is being too lenient.

Pattern Analysis:
{fp_patterns}

Specific Examples ({min(len(false_positives), 20)} of {len(false_positives)}):
{fp_lines}
{reference_examples_text}
{logic_mismatch_analysis}

INSTRUCTIONS:
1. Analyze the error patterns above carefully.
2. Identify the root causes of both false negatives and false positives.
3. Revise the USER PROMPT to address BOTH issues simultaneously.
4. Make the prompt more specific about edge cases.
5. Add clarifications that will help the model distinguish between similar-looking valid and invalid scenarios.
6. Ensure the prompt guides the model to achieve {target_recall}% recall AND {target_precision}% precision.
7. IMPORTANT: Include a ### REFERENCE EXAMPLES section in your revised prompt. This section should reference the 2 most clear False Negatives and 2 most clear False Positives shown above. For each example, include a Correct Logic block explaining why the previous model reason was wrong and what the correct decision is.
8. CRITICAL - Logic Mismatch Analysis: If the Logic Mismatch Analysis above indicates:
   - 'SEEING FEATURES BUT WRONG DECISION': Sharpen the Parking Rules section to make explicit connections between detected features (ramps, blocking, posture) and PASS/FAIL decisions. Add conditional logic like 'If a ramp is detected within X feet, then FAIL' or 'If is_blocking_path is true, then FAIL'.
   - 'MISSING FEATURES': Add a ### VISUAL SCRUTINY CHECKLIST section before the Parking Rules that instructs the model to systematically check for: nearby ramps/doors/crosswalks, vehicle posture, path blocking, and prohibited zones. Make this a mandatory step-by-step process.

Output ONLY the complete revised USER PROMPT (including ### VISUAL SCRUTINY CHECKLIST if needed, and the ### REFERENCE EXAMPLES section).
""".strip()

    # Use cheaper text-only model for optimizer (no vision needed)
    optimizer_model = OPTIMIZER_MODEL_ID if 'OPTIMIZER_MODEL_ID' in globals() else MODEL_ID
    # Use lower temperature for optimizer (more stable, less prompt drift)
    optimizer_temp = OPTIMIZER_TEMPERATURE if 'OPTIMIZER_TEMPERATURE' in globals() else 0.0
    logger.debug(f"Using optimizer model: {optimizer_model}, temperature: {optimizer_temp}")
    
    payload = {
        "model": optimizer_model,
        "temperature": optimizer_temp,
        "messages": [
            {"role": "system", "content": (system_prompt or "").strip()},
            {"role": "user", "content": prompt},
        ],
    }

    try:
        logger.debug(f"Calling optimizer LLM with model: {optimizer_model}")
        data = post_with_retries(url, headers, payload)
        
        # Extract and track cost for optimizer call
        usage = data.get("usage", {})
        if usage:
            optimizer_model = OPTIMIZER_MODEL_ID if 'OPTIMIZER_MODEL_ID' in globals() else MODEL_ID
            cost = _calculate_cost(usage, optimizer_model)
            _track_cost(cost, "optimizer_calls")
            # Enhanced logging with model and token details
            logger.info(f"Optimizer call: model={optimizer_model}, tokens={usage.get('prompt_tokens', 0)}/{usage.get('completion_tokens', 0)}, cost=${cost:.4f} (ESTIMATE)")
        
        if "choices" not in data or len(data["choices"]) == 0:
            error_msg = f"Invalid response structure from optimizer: {data}"
            logger.error(error_msg)
            raise ValueError(error_msg)
        
        new_prompt = (data["choices"][0]["message"]["content"] or "").strip()
        if not new_prompt:
            error_msg = "Empty prompt returned from optimizer step"
            logger.error(error_msg)
            raise RuntimeError(error_msg)
        
        # Post-process to remove any filenames, URLs, or specific image references
        # This is a safety measure in case the optimizer includes them despite instructions
        import re
        # Remove lines containing filenames (common patterns)
        lines = new_prompt.split('\n')
        cleaned_lines = []
        for line in lines:
            # Skip lines that look like filenames or contain file paths
            if re.search(r'(Filename|filename|File:|file:)\s*:', line, re.IGNORECASE):
                # This is a filename line, skip it
                logger.warning(f"Removed filename line from generated prompt: {line[:80]}")
                continue
            # Remove any .png, .jpg, .jpeg references in the line
            if re.search(r'\.(png|jpg|jpeg|PNG|JPG|JPEG)', line):
                # Clean the line by removing the filename part
                cleaned_line = re.sub(r'[^\s]*\.(png|jpg|jpeg|PNG|JPG|JPEG)[^\s]*', '', line)
                cleaned_line = re.sub(r'Filename:\s*[^\n]*', '', cleaned_line, flags=re.IGNORECASE)
                if cleaned_line.strip():
                    cleaned_lines.append(cleaned_line)
                else:
                    logger.warning(f"Removed line with filename reference: {line[:80]}")
                continue
            # Remove any URLs
            if re.search(r'https?://', line):
                cleaned_line = re.sub(r'https?://[^\s]+', '', line)
                if cleaned_line.strip():
                    cleaned_lines.append(cleaned_line)
                else:
                    logger.warning(f"Removed line with URL: {line[:80]}")
                continue
            cleaned_lines.append(line)
        
        new_prompt = '\n'.join(cleaned_lines).strip()
        
        logger.info(f"New prompt generated (length: {len(new_prompt)} chars)")
        logger.info(f"Identified {len(reference_examples)} reference examples for few-shot learning")
        return new_prompt, reference_examples
    except Exception as e:
        logger.error(f"Error in propose_next_user_prompt: {type(e).__name__}: {str(e)}")
        raise


# -------------------- Evaluation + optimization loop --------------------

@dataclass
class ImageItem:
    file: str
    gt: str
    data_url: str = ""  # Will be computed lazily if empty
    evaluation_message: str = ""  # Optional: model's reasoning from CSV
    
    def get_data_url(self) -> str:
        """Lazy load data URL - compute only when needed"""
        if not self.data_url:
            # Check if file exists (now using persistent paths, so should exist)
            # URLs are always valid, so only check local files
            if not self.file.startswith(('http://', 'https://')):
                if not os.path.exists(self.file):
                    raise FileNotFoundError(
                        f"File no longer exists: {self.file}\n"
                        f"This should not happen with persistent uploads. File may have been manually deleted."
                    )
            self.data_url = to_data_url(self.file)
        return self.data_url

def evaluate_single_item(
    item: ImageItem,
    system_prompt: str,
    user_prompt: str,
    idx: int,
    total: int,
    reference_examples: Optional[List[Dict[str, Any]]] = None
) -> Dict[str, Any]:
    """Evaluate a single image item (used for parallel processing)"""
    # Check for stop request
    if stop_event.is_set():
        raise InterruptedError("Optimization stopped by user")
    
    try:
        logger.debug(f"Evaluating item {idx}/{total}: {os.path.basename(item.file)}")
        out = openrouter_classify(system_prompt, user_prompt, item.get_data_url(), reference_examples=reference_examples)
        pred = out["decision"]  # This is now the deterministic decision
        model_pred = out.get("model_decision", pred)  # Keep original for comparison
        reason = out.get("reason", "")
        visual_checklist = out.get("visual_checklist", {})
        rule_validation = out.get("rule_validation", {})
        logger.debug(f"Item {idx} result: {pred} (deterministic), model said: {model_pred}")
        return {
            "file": os.path.basename(item.file),
            "file_path": item.file,  # Store full path for download functionality
            "gt": item.gt,
            "pred": pred,  # Deterministic decision from rules
            "model_pred": model_pred,  # Original model decision (for analysis)
            "reason": reason,
            "visual_checklist": visual_checklist,
            "rule_validation": rule_validation,
            "error": False
        }
    except InterruptedError:
        raise  # Re-raise stop requests
    except Exception as e:
        logger.warning(f"Error evaluating item {idx} ({os.path.basename(item.file)}): {type(e).__name__}: {str(e)}")
        return {
            "file": os.path.basename(item.file),
            "file_path": item.file,  # Store full path for download functionality
            "gt": item.gt,
            "pred": "FAIL",  # conservative fallback
            "reason": f"ERROR: {str(e)[:180]}",
            "visual_checklist": {},
            "rule_validation": {},
            "error": True
        }

def evaluate_prompt(
    system_prompt: str,
    user_prompt: str,
    items: List[ImageItem],
    progress: Optional[gr.Progress] = None,
    prefix: str = "Eval",
    ux_focus_pct: float = 0.0,
    reference_examples: Optional[List[Dict[str, Any]]] = None
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    logger.info(f"Evaluating prompt on {len(items)} items (prefix: {prefix}) using {MAX_WORKERS} workers")
    rows: List[Dict[str, Any]] = []
    errors = 0
    total = len(items)
    
    # Use parallel processing for faster evaluation
    completed = 0
    rows_dict = {}  # Use dict to maintain order
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks
        future_to_idx = {
            executor.submit(evaluate_single_item, it, system_prompt, user_prompt, idx, total, reference_examples): idx
            for idx, it in enumerate(items, start=1)
        }
        
        # Process completed tasks as they finish
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            completed += 1
            
            # Check for stop request
            if stop_event.is_set():
                logger.warning(f"Stop requested during evaluation at item {completed}/{total}")
                # Cancel remaining tasks
                for f in future_to_idx:
                    f.cancel()
                raise InterruptedError("Optimization stopped by user")
            
            # Update progress
            if progress is not None:
                progress(completed / max(total, 1), desc=f"{prefix}: {completed}/{total}")
            
            try:
                result = future.result(timeout=OPENROUTER_TIMEOUT_SECONDS + 10)  # Add small buffer
                rows_dict[idx] = result
                if result.get("error", False):
                    errors += 1
            except InterruptedError:
                raise
            except TimeoutError as e:
                logger.error(f"Timeout processing item {idx}: {e}")
                rows_dict[idx] = {
                    "file": os.path.basename(items[idx-1].file),
                    "gt": items[idx-1].gt,
                    "pred": "FAIL",
                    "reason": f"ERROR: Timeout after {OPENROUTER_TIMEOUT_SECONDS}s",
                    "visual_checklist": {},
                    "rule_validation": {},
                    "error": True
                }
                errors += 1
            except Exception as e:
                logger.error(f"Unexpected error processing item {idx}: {type(e).__name__}: {e}", exc_info=True)
                rows_dict[idx] = {
                    "file": os.path.basename(items[idx-1].file),
                    "file_path": items[idx-1].file,  # Store full path for download functionality
                    "gt": items[idx-1].gt,
                    "pred": "FAIL",
                    "reason": f"ERROR: {str(e)[:180]}",
                    "visual_checklist": {},
                    "rule_validation": {},
                    "error": True
                }
                errors += 1
    
    # Reconstruct rows in original order
    # Ensure we have results for all items (fill missing ones)
    rows = []
    for idx in range(1, total + 1):
        if idx in rows_dict:
            row = rows_dict[idx].copy()
            row.pop("error", None)  # Remove error flag before returning
            rows.append(row)
        else:
            # Missing result - should not happen, but handle gracefully
            logger.warning(f"Missing result for item {idx}, creating fallback")
            rows.append({
                "file": os.path.basename(items[idx-1].file),
                "file_path": items[idx-1].file,  # Store full path for download functionality
                "gt": items[idx-1].gt,
                "pred": "FAIL",
                "reason": "ERROR: Missing result",
                "visual_checklist": {},
                "rule_validation": {},
            })
            errors += 1
    
    if progress is not None:
        progress(1.0, desc=f"{prefix}: done")
    
    logger.info(f"{prefix}: Evaluation completed. Processed {len(rows)} items, {errors} errors")
    logger.info(f"{prefix}: Results - TP: {sum(1 for r in rows if r.get('gt') == 'PASS' and r.get('pred') == 'PASS')}, "
                f"FN: {sum(1 for r in rows if r.get('gt') == 'PASS' and r.get('pred') == 'FAIL')}, "
                f"FP: {sum(1 for r in rows if r.get('gt') == 'FAIL' and r.get('pred') == 'PASS')}, "
                f"TN: {sum(1 for r in rows if r.get('gt') == 'FAIL' and r.get('pred') == 'FAIL')}")

    metrics = compute_metrics(rows, ux_focus_pct=ux_focus_pct)
    logger.info(f"{prefix}: Metrics computed - Recall: {metrics.get('recall_pass_pct', 0):.1f}%, Precision: {metrics.get('precision_pass_pct', 0):.1f}%")
    metrics["invalid_json_or_error_count"] = errors
    logger.info(f"Evaluation complete: {metrics} (errors: {errors})")
    return rows, metrics

def better(a: Dict[str, Any], b: Dict[str, Any]) -> Dict[str, Any]:
    ma = a["metrics"]
    mb = b["metrics"]
    # Compare by weighted score (higher is better)
    score_a = ma.get("weighted_score", 0)
    score_b = mb.get("weighted_score", 0)
    return a if score_a >= score_b else b

def run_optimizer(
    files: List[str],
    market: str = "",
    ux_focus_pct: float = 0.0,
    progress: Optional[gr.Progress] = None,
    preprocessed_uploaded_files: Optional[List[Dict[str, Any]]] = None,
    custom_system_prompt: str = "",
    custom_user_prompt: str = "",
):
    optimizer_start_time = time.time()
    # Reset stop event at start (important for multiple runs)
    stop_event.clear()
    # Reset cost tracker at start of each run
    reset_cost_tracker()
    logger.info("Stop event cleared at start of run_optimizer")
    logger.info("Cost tracker reset")
    
    logger.info("=" * 80)
    logger.info("Starting optimization run")
    logger.info("=" * 80)
    logger.info(f"Time at start of run_optimizer: {time.time() - optimizer_start_time:.3f}s")
    
    try:
        items: List[ImageItem] = []
        pass_n = 0
        fail_n = 0
        bad = 0
        base_photos_count = 0
        market_name = ""
        
        # Process uploaded files FIRST (they're already pre-processed and ready)
        # This prevents blocking on slow CSV/base photo downloads
        # Handle None or empty files
        if files is None:
            files = []
        elif not isinstance(files, (list, tuple)):
            files = [files] if files else []
        
        uploaded_count = len(files) if files else 0
        logger.info(f"Processing {uploaded_count} uploaded files (files type: {type(files)})")
        
        if files:
            if preprocessed_uploaded_files:
                # Use pre-processed files (already copied and converted to data URLs)
                logger.info(f"Using pre-processed uploaded files (data URLs already in memory)")
                successful_uploads = 0
                for processed_file in preprocessed_uploaded_files:
                    try:
                        file_path = processed_file["file"]
                        data_url = processed_file.get("data_url", "")
                        
                        # Infer label from filename
                        gt = infer_label(file_path)
                        
                        if gt == "PASS":
                            pass_n += 1
                        else:
                            fail_n += 1
                        
                        # Add to items list with pre-computed data URL
                        items.append(ImageItem(
                            file=file_path,
                            gt=gt,
                            data_url=data_url  # Pre-computed and held in memory
                        ))
                        successful_uploads += 1
                        logger.debug(f"Added pre-processed uploaded item: {os.path.basename(file_path)} -> {gt}")
                    except Exception as e:
                        bad += 1
                        logger.warning(f"Error adding pre-processed file {processed_file.get('original_name', 'unknown')}: {e}")
                
                logger.info(f"Added {successful_uploads}/{len(preprocessed_uploaded_files)} pre-processed uploaded files to items list")
            else:
                # Fallback: process files now (shouldn't happen if wrapper works correctly)
                logger.warning("No pre-processed files provided, processing files now (this is slower)")
                file_processing_start = time.time()
                progress(0.0, desc="Processing uploaded photos...")
                
                def process_uploaded_file(args):
                    idx, f = args
                    try:
                        if not f or not os.path.exists(f):
                            return {"success": False, "error": "File not found", "idx": idx}
                        
                        gt = infer_label(f)
                        data_url = to_data_url(f)
                        
                        return {
                            "success": True,
                            "file": f,
                            "gt": gt,
                            "data_url": data_url,
                            "idx": idx
                        }
                    except Exception as e:
                        return {"success": False, "error": str(e), "idx": idx}
                
                try:
                    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, uploaded_count)) as executor:
                        uploaded_results = list(executor.map(process_uploaded_file, enumerate(files)))
                except Exception as e:
                    logger.error(f"Error in parallel file processing: {e}")
                    uploaded_results = [process_uploaded_file((idx, f)) for idx, f in enumerate(files)]
                
                successful_uploads = 0
                for result in sorted(uploaded_results, key=lambda x: x.get("idx", 0)):
                    if result["success"]:
                        if result["gt"] == "PASS":
                            pass_n += 1
                        else:
                            fail_n += 1
                        items.append(ImageItem(
                            file=result["file"],
                            gt=result["gt"],
                            data_url=result["data_url"]
                        ))
                        successful_uploads += 1
                    else:
                        bad += 1
                
                elapsed = time.time() - file_processing_start
                logger.info(f"Processed {successful_uploads}/{uploaded_count} uploaded files in {elapsed:.2f} seconds")
        
        # Load base photos from CSV ONLY if market is provided (skip search if no market)
        # Do this AFTER uploaded files so uploaded files aren't blocked by slow URL downloads
        if market and market.strip():
            market = market.strip()
            market_name = market
            progress(0.0, desc="Loading base photos from CSV...")
            logger.info(f"Loading base photos for market: {market}")
            base_photos, csv_error = load_base_photos_from_csv(market)
            
            if csv_error:
                error_msg = f"Error loading base photos: {csv_error}"
                logger.error(error_msg)
                raise ValueError(error_msg)
            
            if base_photos:
                base_photos_count = len(base_photos)
                logger.info(f"Loaded {base_photos_count} base photos for market '{market}'")
                
                # Pre-download and convert base photos in parallel for better performance
                # This avoids slow downloads during evaluation (network bottleneck)
                progress(0.0, desc="Pre-downloading base photos from URLs...")
                logger.info(f"Pre-downloading {base_photos_count} base photos from URLs (this may take a moment)")
                download_start = time.time()
                
                def process_base_photo(photo):
                    try:
                        gt = photo['evaluation_status']  # PASS or FAIL from CSV
                        # Pre-download and convert to data URL now (not lazy) - do it in parallel
                        data_url = to_data_url(photo['image_url'])
                        return {
                            "file": photo['image_url'],
                            "gt": gt,
                            "data_url": data_url,
                            "evaluation_message": photo['evaluation_message'],
                            "success": True
                        }
                    except Exception as e:
                        logger.warning(f"Error processing base photo {photo.get('image_url', 'unknown')}: {e}")
                        return {"error": str(e), "success": False}
                
                # Process base photos in parallel (downloads happen concurrently)
                with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, base_photos_count)) as executor:
                    base_results = list(executor.map(process_base_photo, base_photos))
                
                # Add successful base photos
                successful_downloads = 0
                for result in base_results:
                    if result["success"]:
                        if result["gt"] == "PASS":
                            pass_n += 1
                        else:
                            fail_n += 1
                        items.append(ImageItem(
                            file=result["file"],
                            gt=result["gt"],
                            data_url=result["data_url"],  # Pre-computed, not lazy
                            evaluation_message=result["evaluation_message"]
                        ))
                        successful_downloads += 1
                        logger.debug(f"Added base photo: {result['file'][:50]}... -> {result['gt']}")
                    else:
                        bad += 1
                
                download_elapsed = time.time() - download_start
                logger.info(f"Pre-downloaded {successful_downloads}/{base_photos_count} base photos in {download_elapsed:.2f} seconds")
                if download_elapsed > 10:
                    logger.warning(f"⚠️ Base photo downloads took {download_elapsed:.2f}s - network may be slow")

        dataset_prep_end = time.time()
        logger.info(f"Dataset prepared: total={len(items)}, base_photos={base_photos_count}, uploaded={uploaded_count}, PASS={pass_n}, FAIL={fail_n}, bad={bad}")
        logger.info(f"Total dataset preparation time: {dataset_prep_end - optimizer_start_time:.2f}s")

        if not items:
            if market:
                error_msg = f"No valid images found for market '{market}'. Check CSV file or upload images (filenames must start with PASS_ or FAIL_)."
            else:
                error_msg = "No valid images found. Upload images (filenames must start with PASS_ or FAIL_) or enter a market name."
            logger.error(error_msg)
            raise ValueError(error_msg)
        if pass_n < MIN_PASS_IMAGES_REQUIRED:
            error_msg = (
                f"Not enough PASS images to evaluate recall reliably "
                f"(found {pass_n}, need at least {MIN_PASS_IMAGES_REQUIRED})."
            )
            logger.error(error_msg)
            raise ValueError(error_msg)

        # Use custom prompts if provided, otherwise use DEFAULT prompts, otherwise use fallback
        if custom_system_prompt and custom_system_prompt.strip():
            system_prompt = custom_system_prompt.strip()
            logger.info("Using custom system prompt from UI")
        else:
            try:
                system_prompt = DEFAULT_SYSTEM_PROMPT
                logger.info("Using DEFAULT_SYSTEM_PROMPT")
            except NameError:
                logger.warning("DEFAULT_SYSTEM_PROMPT not found, using fallback")
        
        if custom_user_prompt and custom_user_prompt.strip():
            base_user_prompt = custom_user_prompt.strip()
            logger.info("Using custom user prompt from UI")
        else:
            try:
                base_user_prompt = DEFAULT_BASE_USER_PROMPT
                logger.info("Using DEFAULT_BASE_USER_PROMPT")
            except NameError:
                logger.warning("DEFAULT_BASE_USER_PROMPT not found, using fallback")
        
        # Fallback prompts if neither custom nor DEFAULT are available
        if 'system_prompt' not in locals() or not system_prompt:
            system_prompt = """You are a micromobility parking enforcement officer.
Analyze this parking photo of a shared e-scooter or bicycle.
Rate it as PASS or FAIL and provide feedback accordingly.

Condense your feedback into a single, short sentence.
If FAIL, make it actionable so the rider knows what they need to improve.""".strip()
        
        if 'base_user_prompt' not in locals() or not base_user_prompt:
            base_user_prompt = """Apply the following rules:

1. The vehicle must be fully visible in the photo. It cannot be cut off. The photo should include enough surrounding space to judge whether the vehicle is parked appropriately.
2. Vehicles must not block sidewalks or pedestrian paths.
3. Parking on sidewalks is only ok if the vehicle is fully to the side and does not obstruct pedestrian flow.
4. Parking in the furniture zone of the sidewalk/street is ok and better than if parked in the middle of the sidewalk
5. Parking in designated micromobility spaces is preferred, but optional. Designated parking spaces are usually marked with paint on the ground or clear sign posts or other signage.
6. Vehicles must not block streets, driveways, public transport stations or entrances
7. Parking near crosswalks, ramps, or handicap-accessible areas is not allowed.
8. The vehicle must be upright and not tipped over.

Only refer to the brand of the scooter or bike if you are absolutely sure you detected the brand correctly (Bird, Spin).""".strip()

        logger.info(f"System prompt length: {len(system_prompt)} chars")
        logger.info(f"Base user prompt length: {len(base_user_prompt)} chars")

        history: List[Dict[str, Any]] = []
        
        # Helper function to build hard set from evaluation results
        def build_hard_set(rows: List[Dict[str, Any]], all_items: List[ImageItem], target_size: int = 15, random_count: int = 3) -> List[ImageItem]:
            """Build a hard set containing all FN+FP plus random samples for diversity."""
            # Get all false negatives and false positives
            false_neg = [r for r in rows if r["gt"] == "PASS" and r["pred"] == "FAIL"]
            false_pos = [r for r in rows if r["gt"] == "FAIL" and r["pred"] == "PASS"]
            
            # Create mapping from filename to ImageItem
            items_by_filename = {os.path.basename(item.file): item for item in all_items}
            
            hard_set_items = []
            hard_set_filenames = set()
            
            # Add all false negatives
            for fn in false_neg:
                filename = os.path.basename(fn.get("file", ""))
                if filename and filename in items_by_filename:
                    if filename not in hard_set_filenames:
                        hard_set_items.append(items_by_filename[filename])
                        hard_set_filenames.add(filename)
            
            # Add all false positives
            for fp in false_pos:
                filename = os.path.basename(fp.get("file", ""))
                if filename and filename in items_by_filename:
                    if filename not in hard_set_filenames:
                        hard_set_items.append(items_by_filename[filename])
                        hard_set_filenames.add(filename)
            
            # Add random samples for diversity (if we haven't reached target size)
            remaining_needed = max(0, target_size - len(hard_set_items))
            if remaining_needed > 0:
                import random
                # Get items not already in hard set
                available_items = [item for item in all_items if os.path.basename(item.file) not in hard_set_filenames]
                if available_items:
                    random_samples = random.sample(available_items, min(remaining_needed, len(available_items)))
                    for item in random_samples:
                        hard_set_items.append(item)
                        hard_set_filenames.add(os.path.basename(item.file))
            
            logger.info(f"Built hard set: {len(hard_set_items)} items ({len(false_neg)} FN, {len(false_pos)} FP, {len(hard_set_items) - len(false_neg) - len(false_pos)} random)")
            return hard_set_items

        # Iteration 0 (base prompt) - ALWAYS run on all images
        iter0_start = time.time()
        logger.info("Starting iteration 0 (base prompt evaluation)")
        if progress is not None:
            progress(0.0, desc="Iteration 0: evaluating base prompt...")
        try:
            rows0, m0 = evaluate_prompt(system_prompt, base_user_prompt, items, progress=progress, prefix="Iter 0", ux_focus_pct=ux_focus_pct)
            iter0_end = time.time()
            logger.info(f"Iteration 0 completed in {iter0_end - iter0_start:.2f} seconds")
            history.append({"iteration": 0, "user_prompt": base_user_prompt, "metrics": m0, "rows": rows0})
            best = history[0]
            current_prompt = base_user_prompt
            logger.info(f"Iteration 0 complete: {m0}")
        except Exception as e:
            logger.error(f"Error in iteration 0: {type(e).__name__}: {str(e)}", exc_info=True)
            raise
        
        # Check if we should skip optimization (evaluation only mode)
        evaluation_only = EVALUATION_ONLY if 'EVALUATION_ONLY' in globals() else False
        
        if evaluation_only:
            logger.info("=" * 80)
            logger.info("📊 EVALUATION ONLY MODE: Skipping optimization iterations")
            logger.info("=" * 80)
            logger.info(f"Baseline Evaluation Results:")
            logger.info(f"  Recall: {m0['recall_pass_pct']}%")
            logger.info(f"  Precision: {m0['precision_pass_pct']}%")
            logger.info(f"  Weighted Score: {m0.get('weighted_score', 0)}%")
            logger.info(f"  TP={m0['tp']}, FN={m0['fn']}, FP={m0['fp']}, TN={m0['tn']}")
            logger.info(f"  Total images: {len(items)}")
            logger.info("=" * 80)
            # Set best to baseline and skip optimization loop
            best = history[0]
            best_metrics = best["metrics"]
            best_prompt = best["user_prompt"]
            best_rows = best["rows"]
        else:
            # Build hard set after baseline (if hard set optimization is enabled)
            hard_set_items = items  # Default to all items
            use_hard_set = USE_HARD_SET_OPTIMIZATION if 'USE_HARD_SET_OPTIMIZATION' in globals() else False
            if use_hard_set:
                hard_set_size = HARD_SET_SIZE if 'HARD_SET_SIZE' in globals() else 15
                random_count = HARD_SET_RANDOM_COUNT if 'HARD_SET_RANDOM_COUNT' in globals() else 3
                hard_set_items = build_hard_set(rows0, items, target_size=hard_set_size, random_count=random_count)
                logger.info(f"🔧 Hard set optimization enabled: using {len(hard_set_items)} items for optimization (out of {len(items)} total)")

            # Iterations 1..OPTIMIZATION_ITERATIONS (use hard set if enabled)
            for i in range(1, OPTIMIZATION_ITERATIONS + 1):
                # Check for stop request before starting iteration
                if stop_event.is_set():
                    logger.warning(f"Stop requested before iteration {i}")
                    raise InterruptedError("Optimization stopped by user")
                
                logger.info(f"Starting iteration {i}")
                try:
                    progress(0.0, desc=f"Iteration {i}: building edge cases...")
                    last_rows = history[-1]["rows"]
                    false_neg = [r for r in last_rows if r["gt"] == "PASS" and r["pred"] == "FAIL"]
                    false_pos = [r for r in last_rows if r["gt"] == "FAIL" and r["pred"] == "PASS"]
                    logger.info(f"Iteration {i}: Found {len(false_neg)} false negatives, {len(false_pos)} false positives")
                    if use_hard_set:
                        logger.info(f"  (Note: These are from hard set of {len(hard_set_items)} items, not all {len(items)} items)")

                    # Check for stop request
                    if stop_event.is_set():
                        logger.warning(f"Stop requested during iteration {i} (building edge cases)")
                        raise InterruptedError("Optimization stopped by user")

                    progress(0.0, desc=f"Iteration {i}: proposing new prompt...")
                    # Get current metrics from last iteration
                    current_metrics = history[-1]["metrics"]
                    new_prompt, reference_examples = propose_next_user_prompt(
                        system_prompt=system_prompt,
                        current_user_prompt=current_prompt,
                        false_negatives=false_neg,
                        false_positives=false_pos,
                        current_metrics=current_metrics,
                        items=items,
                        ux_focus_pct=ux_focus_pct,
                    )

                    # Check for stop request
                    if stop_event.is_set():
                        logger.warning(f"Stop requested during iteration {i} (after prompt proposal)")
                        raise InterruptedError("Optimization stopped by user")

                    iter_start = time.time()
                    # Use hard set for optimization iterations if enabled
                    eval_items = hard_set_items if use_hard_set else items
                    eval_prefix = f"Iter {i} (hard set)" if use_hard_set else f"Iter {i}"
                    progress(0.0, desc=f"Iteration {i}: evaluating new prompt on {len(eval_items)} items...")
                    rows_i, m_i = evaluate_prompt(system_prompt, new_prompt, eval_items, progress=progress, prefix=eval_prefix, ux_focus_pct=ux_focus_pct, reference_examples=reference_examples)
                    iter_end = time.time()
                    logger.info(f"Iteration {i} completed in {iter_end - iter_start:.2f} seconds (evaluated {len(eval_items)} items)")
                    
                    # Compare with baseline to see if we're improving
                    baseline_metrics = history[0]["metrics"]
                    baseline_weighted = baseline_metrics.get('weighted_score', 0)
                    current_weighted = m_i.get('weighted_score', 0)
                    improvement = current_weighted - baseline_weighted
                    if use_hard_set:
                        logger.info(f"  Hard set metrics: weighted={current_weighted}% (baseline on all items: {baseline_weighted}%)")
                        logger.info(f"  ⚠️  Note: Hard set has {len(eval_items)} items vs {len(items)} in baseline - metrics not directly comparable")
                    else:
                        logger.info(f"  Metrics vs baseline: weighted={current_weighted}% (baseline: {baseline_weighted}%, delta: {improvement:+.2f}%)")
                    
                    candidate = {"iteration": i, "user_prompt": new_prompt, "metrics": m_i, "rows": rows_i}
                    history.append(candidate)
                    
                    # Check for early stopping if both targets are reached
                    recall_i = m_i.get('recall_pass_pct', 0)
                    precision_i = m_i.get('precision_pass_pct', 0)
                    if recall_i >= EARLY_STOP_THRESHOLD and precision_i >= EARLY_STOP_THRESHOLD:
                        logger.info(f"🎯 TARGET ACHIEVED! Recall: {recall_i}%, Precision: {precision_i}% - Stopping early")
                        break
                    logger.info(f"Iteration {i} complete: {m_i}")

                    # When using hard set, don't compare with baseline (different datasets)
                    # We'll evaluate all candidates on full dataset at the end
                    if not use_hard_set:
                        # Normal mode: compare directly
                        best = better(best, candidate)
                    else:
                        # Hard set mode: keep all candidates, will evaluate best on full dataset later
                        # Don't update 'best' here - metrics are from different dataset sizes
                        logger.info(f"  (Hard set mode: not comparing with baseline - will evaluate on full dataset at end)")
                    
                    current_prompt = new_prompt
                    
                    # Rebuild hard set mid-run (after iteration 2) to adapt to policy changes
                    if use_hard_set and i == 2:
                        logger.info("🔄 Rebuilding hard set after iteration 2 to adapt to policy changes...")
                        hard_set_size = HARD_SET_SIZE if 'HARD_SET_SIZE' in globals() else 15
                        random_count = HARD_SET_RANDOM_COUNT if 'HARD_SET_RANDOM_COUNT' in globals() else 3
                        hard_set_items = build_hard_set(rows_i, items, target_size=hard_set_size, random_count=random_count)
                        logger.info(f"🔧 Hard set rebuilt: {len(hard_set_items)} items")
                    
                    # Early stopping: if no improvement on hard set, stop early
                    if use_hard_set and i > 1:
                        prev_metrics = history[-2]["metrics"]
                        prev_weighted = prev_metrics.get('weighted_score', 0)
                        curr_weighted = m_i.get('weighted_score', 0)
                        if curr_weighted <= prev_weighted:
                            logger.info(f"⚠️ No improvement on hard set (weighted score: {prev_weighted} -> {curr_weighted}), stopping early")
                            break
                except InterruptedError:
                    # Re-raise interruption errors
                    raise
                except Exception as e:
                    logger.error(f"Error in iteration {i}: {type(e).__name__}: {str(e)}", exc_info=True)
                    raise
        
        # Final evaluation on ALL images (if hard set optimization was used and not in evaluation-only mode)
        # Skip this section entirely if in evaluation-only mode
        if not evaluation_only:
            # Need to evaluate all candidate prompts on full dataset to find the true best
            if use_hard_set and len(hard_set_items) < len(items):
                logger.info("=" * 80)
                logger.info("Evaluating all candidate prompts on FULL dataset to find true best")
                logger.info("=" * 80)
                
                # Collect all candidate prompts (iteration 0 + optimized iterations)
                candidates_to_evaluate = []
                for h in history:
                    if h["iteration"] != "final":  # Skip any existing final evaluations
                        candidates_to_evaluate.append(h)
                
                # Evaluate each candidate on full dataset
                best_candidate = None
                best_score = -1
                
                for candidate in candidates_to_evaluate:
                    iter_num = candidate["iteration"]
                    candidate_prompt = candidate["user_prompt"]
                    logger.info(f"Evaluating iteration {iter_num} prompt on full dataset ({len(items)} items)...")
                    
                    final_start = time.time()
                    progress(0.0, desc=f"Evaluating iteration {iter_num} on all images...")
                    # Use FINAL_EVAL_TEMPERATURE for final evaluation (more realistic)
                    final_eval_temp = FINAL_EVAL_TEMPERATURE if 'FINAL_EVAL_TEMPERATURE' in globals() else 0.2
                    # Temporarily override MODEL_TEMPERATURE for this evaluation
                    original_temp = MODEL_TEMPERATURE
                    try:
                        # Set temp to final eval temp
                        import sys
                        if 'MODEL_TEMPERATURE' in globals():
                            globals()['MODEL_TEMPERATURE'] = final_eval_temp
                        final_rows, final_metrics = evaluate_prompt(
                            system_prompt, 
                            candidate_prompt, 
                            items, 
                            progress=progress, 
                            prefix=f"Final-{iter_num}", 
                            ux_focus_pct=ux_focus_pct
                        )
                    finally:
                        # Restore original temperature
                        if 'MODEL_TEMPERATURE' in globals():
                            globals()['MODEL_TEMPERATURE'] = original_temp
                    final_end = time.time()
                    
                    final_weighted = final_metrics.get('weighted_score', 0)
                    logger.info(f"Iteration {iter_num} on full dataset: weighted_score={final_weighted}% (took {final_end - final_start:.2f}s)")
                    
                    # Update the history entry with full dataset metrics (so history dataframe shows accurate metrics)
                    for h in history:
                        if h["iteration"] == iter_num:
                            h["metrics"] = final_metrics  # Update with full dataset metrics
                            h["rows"] = final_rows  # Update with full dataset rows
                            logger.info(f"  Updated iteration {iter_num} metrics in history with full dataset results")
                            break
                    
                    # Track best candidate
                    if final_weighted > best_score:
                        best_score = final_weighted
                        best_candidate = {
                            "iteration": iter_num,
                            "user_prompt": candidate_prompt,
                            "metrics": final_metrics,
                            "rows": final_rows
                        }
                
                # Use the best candidate from full dataset evaluation
                # But only if it's not significantly worse than baseline
                baseline_metrics = history[0]["metrics"]
                baseline_weighted = baseline_metrics.get('weighted_score', 0)
                
                if best_candidate:
                    improvement = best_score - baseline_weighted
                    if improvement < -5.0:  # If more than 5% worse, stick with baseline
                        logger.warning(f"⚠️ Best optimized prompt ({best_score}%) is {abs(improvement):.1f}% worse than baseline ({baseline_weighted}%)")
                        logger.warning(f"   Using baseline prompt (iteration 0) instead")
                        best = history[0]  # Use baseline
                    else:
                        best = best_candidate
                        history.append({"iteration": f"final-{best_candidate['iteration']}", "user_prompt": best_candidate["user_prompt"], "metrics": best_candidate["metrics"], "rows": best_candidate["rows"]})
                        logger.info(f"✅ Best prompt: iteration {best_candidate['iteration']} with weighted_score={best_score}% on full dataset (vs baseline: {baseline_weighted}%, improvement: {improvement:+.1f}%)")
                else:
                    logger.warning("No candidates evaluated - using baseline")
                    best = history[0]
            else:
                # Normal mode (not using hard set) - best is already set during iterations
                pass
        else:
            # Evaluation-only mode: best is already set above
            pass

        best_metrics = best["metrics"]
        best_prompt = best["user_prompt"]
        best_rows = best["rows"]

        logger.info(f"Optimization complete. Best iteration: {best['iteration']}")
        logger.info(f"Best metrics: {best_metrics}")
        
        # Final save of persistent cache
        _save_persistent_cache()
        logger.info("✅ Saved persistent cache to disk")
        
        # Get and display cost summary with cache stats
        cost_summary = get_cost_summary()
        cache_stats = get_cache_stats()
        cache_hit_rate = 0.0
        if len(history) > 0:
            total_calls = len(history) * len(items)  # Rough estimate
            cache_hits = cache_stats.get('cache_size', 0)
            cache_hit_rate = (cache_hits / max(total_calls, 1)) * 100 if total_calls > 0 else 0
        
        logger.info("=" * 80)
        logger.info("💰 COST SUMMARY (ESTIMATED - actual may vary)")
        logger.info("=" * 80)
        logger.info(f"Vision model calls: ${cost_summary['vision_calls']:.4f}")
        logger.info(f"Optimizer calls: ${cost_summary['optimizer_calls']:.4f}")
        logger.info(f"TOTAL COST: ${cost_summary['total']:.4f} (ESTIMATE)")
        logger.info(f"Cache entries: {cache_stats.get('cache_size', 0)}, estimated hit rate: {cache_hit_rate:.1f}%")
        logger.info("=" * 80)

        fn = [r for r in best_rows if r["gt"] == "PASS" and r["pred"] == "FAIL"]
        fp = [r for r in best_rows if r["gt"] == "FAIL" and r["pred"] == "PASS"]
        hall_rows = extract_hallucination_rows(best_rows)
        
        # Store full rows with paths for download functionality
        fn_rows_with_paths = fn  # Keep original rows with file_path
        fp_rows_with_paths = fp  # Keep original rows with file_path
        hall_rows_with_paths = hall_rows  # Keep original rows with file_path

        df_history = pd.DataFrame([{
            "iteration": h["iteration"],
            "weighted_score": h["metrics"].get("weighted_score", 0),
            "recall_pass_pct": h["metrics"]["recall_pass_pct"],
            "precision_pass_pct": h["metrics"]["precision_pass_pct"],
            "fp_rate_pct": _compute_fpr_pct(h["metrics"].get("fp", 0), h["metrics"].get("tn", 0)),
            "logic_mismatch_pct": h["metrics"].get("logic_mismatch_pct", 0),
            "tp": h["metrics"]["tp"],
            "fn": h["metrics"]["fn"],
            "fp": h["metrics"]["fp"],
            "tn": h["metrics"]["tn"],
            "invalid_json_or_error_count": h["metrics"].get("invalid_json_or_error_count", 0),
        } for h in history])

        # Ensure DataFrames are properly formatted for Gradio
        df_fn = pd.DataFrame(fn) if fn else pd.DataFrame(columns=["file", "gt", "pred", "reason"])
        df_fp = pd.DataFrame(fp) if fp else pd.DataFrame(columns=["file", "gt", "pred", "reason"])
        df_hall = pd.DataFrame(hall_rows) if hall_rows else pd.DataFrame(columns=["file", "pred", "reason", "checklist_vehicle_posture", "hallucination_issue"])
        
        # Replace NaN values with empty strings for Gradio compatibility
        df_history = df_history.fillna("")
        df_fn = df_fn.fillna("")
        df_fp = df_fp.fillna("")
        df_hall = df_hall.fillna("")
        
        # Ensure all DataFrames have string types for all columns (Gradio requirement)
        for col in df_history.columns:
            df_history[col] = df_history[col].astype(str)
        for col in df_fn.columns:
            df_fn[col] = df_fn[col].astype(str)
        for col in df_fp.columns:
            df_fp[col] = df_fp[col].astype(str)
        for col in df_hall.columns:
            df_hall[col] = df_hall[col].astype(str)

        # Build summary with market and base photo info
        market_info = f"Market: {market_name}\n" if market_name else ""
        base_photos_info = f"Base photos: {base_photos_count}\n" if base_photos_count > 0 else ""
        uploaded_info = f"Uploaded photos: {uploaded_count}\n" if uploaded_count > 0 else ""
        
        # Get cost summary for display
        cost_summary = get_cost_summary()
        
        summary = (
            f"Model: {MODEL_ID} | Temp: {MODEL_TEMPERATURE}\n"
            f"{market_info}{base_photos_info}{uploaded_info}"
            f"Dataset: total={len(items)} PASS={pass_n} FAIL={fail_n} bad={bad}\n"
            f"Optimization Focus: {best_metrics.get('ux_focus_pct', 0)}% UX / {100-best_metrics.get('ux_focus_pct', 0)}% GP\n"
            f"Best iteration: {best['iteration']}\n"
            f"Weighted Score: {best_metrics.get('weighted_score', 0)}%\n"
            f"Recall(PASS): {best_metrics['recall_pass_pct']}%\n"
            f"Precision(PASS): {best_metrics['precision_pass_pct']}%\n"
            f"TP:{best_metrics['tp']} FN:{best_metrics['fn']} FP:{best_metrics['fp']} TN:{best_metrics['tn']}\n"
            f"Invalid/Errors: {best_metrics.get('invalid_json_or_error_count', 0)}\n"
            f"\n💰 COST BREAKDOWN (ESTIMATED):\n"
            f"Vision calls: ${cost_summary['vision_calls']:.4f}\n"
            f"Optimizer calls: ${cost_summary['optimizer_calls']:.4f}\n"
            f"TOTAL: ${cost_summary['total']:.4f} (ESTIMATE - actual may vary)\n"
        )

        progress(1.0, desc="Done")
        logger.info("=" * 80)
        logger.info("Optimization run completed successfully")
        logger.info("=" * 80)
        
        # Ensure all return values are properly formatted
        logger.info(f"Preparing results: summary length={len(summary)}, prompt length={len(best_prompt)}")
        logger.info(f"DataFrames: history={len(df_history)} rows, fn={len(df_fn)} rows, fp={len(df_fp)} rows")
        
        # Validate DataFrames before returning
        try:
            # Convert to dict for Gradio if needed
            if len(df_history) > 0:
                logger.info(f"History DF columns: {list(df_history.columns)}")
            if len(df_fn) > 0:
                logger.info(f"FN DF columns: {list(df_fn.columns)}")
            if len(df_fp) > 0:
                logger.info(f"FP DF columns: {list(df_fp.columns)}")
            
            logger.info("Returning results to Gradio...")
            # Prepare market and photo stats for UI
            market_display = market_name if market_name else "None"
            photo_stats = f"Base photos: {base_photos_count} | Uploaded: {uploaded_count} | Total: {len(items)} | PASS: {pass_n} | FAIL: {fail_n}"
            return summary, best_prompt, df_history, df_fn, df_fp, df_hall, market_display, photo_stats, fn_rows_with_paths, fp_rows_with_paths, hall_rows_with_paths
        except Exception as e:
            logger.error(f"Error preparing return values: {type(e).__name__}: {str(e)}", exc_info=True)
            # Return safe fallback values
            fallback_summary = f"Error preparing results: {str(e)}"
            fallback_df = pd.DataFrame()
            market_display = market_name if 'market_name' in locals() and market_name else "None"
            return fallback_summary, "", fallback_df, fallback_df, fallback_df, fallback_df, market_display, "", [], [], []
        
    except InterruptedError as e:
        logger.warning(f"Optimization interrupted: {str(e)}")
        # Return partial results if available
        if 'history' in locals() and len(history) > 0:
            best_metrics = best["metrics"]
            best_prompt = best["user_prompt"]
            best_rows = best["rows"]
            
            fn = [r for r in best_rows if r["gt"] == "PASS" and r["pred"] == "FAIL"]
            fp = [r for r in best_rows if r["gt"] == "FAIL" and r["pred"] == "PASS"]
            hall_rows = extract_hallucination_rows(best_rows)
            fn_rows_with_paths = fn
            fp_rows_with_paths = fp
            hall_rows_with_paths = hall_rows
            
            df_history = pd.DataFrame([{
                "iteration": h["iteration"],
                "weighted_score": h["metrics"].get("weighted_score", 0),
                "recall_pass_pct": h["metrics"]["recall_pass_pct"],
                "precision_pass_pct": h["metrics"]["precision_pass_pct"],
                "fp_rate_pct": _compute_fpr_pct(h["metrics"].get("fp", 0), h["metrics"].get("tn", 0)),
                "logic_mismatch_pct": h["metrics"].get("logic_mismatch_pct", 0),
                "tp": h["metrics"]["tp"],
                "fn": h["metrics"]["fn"],
                "fp": h["metrics"]["fp"],
                "tn": h["metrics"]["tn"],
                "invalid_json_or_error_count": h["metrics"].get("invalid_json_or_error_count", 0),
            } for h in history])
            
            df_fn = pd.DataFrame(fn) if fn else pd.DataFrame(columns=["file", "gt", "pred", "reason"])
            df_fp = pd.DataFrame(fp) if fp else pd.DataFrame(columns=["file", "gt", "pred", "reason"])
            df_hall = pd.DataFrame(hall_rows) if hall_rows else pd.DataFrame(columns=["file", "pred", "reason", "checklist_vehicle_posture", "hallucination_issue"])
            
            # Replace NaN values with empty strings for Gradio compatibility
            df_history = df_history.fillna("")
            df_fn = df_fn.fillna("")
            df_fp = df_fp.fillna("")
            df_hall = df_hall.fillna("")
            
            # Build summary with market and base photo info
            market_info = f"Market: {market_name}\n" if market_name else ""
            base_photos_info = f"Base photos: {base_photos_count}\n" if base_photos_count > 0 else ""
            uploaded_info = f"Uploaded photos: {uploaded_count}\n" if uploaded_count > 0 else ""
            
            summary = (
                f"Model: {MODEL_ID} | Temp: {MODEL_TEMPERATURE}\n"
                f"{market_info}{base_photos_info}{uploaded_info}"
                f"Dataset: total={len(items)} PASS={pass_n} FAIL={fail_n} bad={bad}\n"
                f"Optimization Focus: {best_metrics.get('ux_focus_pct', 0)}% UX / {100-best_metrics.get('ux_focus_pct', 0)}% GP\n"
                f"Best iteration: {best['iteration']} (STOPPED EARLY)\n"
                f"Weighted Score: {best_metrics.get('weighted_score', 0)}%\n"
                f"Recall(PASS): {best_metrics['recall_pass_pct']}%\n"
                f"Precision(PASS): {best_metrics['precision_pass_pct']}%\n"
                f"TP:{best_metrics['tp']} FN:{best_metrics['fn']} FP:{best_metrics['fp']} TN:{best_metrics['tn']}\n"
                f"Invalid/Errors: {best_metrics.get('invalid_json_or_error_count', 0)}\n"
            )
            market_display = market_name if market_name else "None"
            photo_stats = f"Base photos: {base_photos_count} | Uploaded: {uploaded_count} | Total: {len(items)} | PASS: {pass_n} | FAIL: {fail_n}"
            return summary, best_prompt, df_history, df_fn, df_fp, df_hall, market_display, photo_stats, fn_rows_with_paths, fp_rows_with_paths, hall_rows_with_paths
        else:
            # No results yet, return empty results
            summary = "Optimization stopped before completion. No results available."
            fn_rows_with_paths = []
            fp_rows_with_paths = []
            hall_rows_with_paths = []
            empty_df = pd.DataFrame()
            market_display = market_name if 'market_name' in locals() and market_name else "None"
            return summary, "", empty_df, empty_df, empty_df, empty_df, market_display, "", fn_rows_with_paths, fp_rows_with_paths, hall_rows_with_paths
    except Exception as e:
        logger.error(f"Fatal error in run_optimizer: {type(e).__name__}: {str(e)}", exc_info=True)
        raise


# -------------------- Gradio UI --------------------

def stop_optimization():
    """Stop the optimization process"""
    stop_event.set()
    logger.info("Stop button clicked - optimization will stop at next checkpoint")
    return "⚠️ Stop requested. Optimization will stop at next checkpoint..."

def cleanup_persistent_uploads():
    """Clean up the persistent uploads directory. Can be called at start or end of optimization."""
    persistent_uploads_dir = "./persistent_uploads"
    try:
        if os.path.exists(persistent_uploads_dir):
            # Try to remove individual files first (more reliable than rmtree on Windows)
            try:
                for filename in os.listdir(persistent_uploads_dir):
                    file_path = os.path.join(persistent_uploads_dir, filename)
                    try:
                        if os.path.isfile(file_path):
                            os.remove(file_path)
                    except Exception as e:
                        logger.debug(f"Could not remove file {filename}: {e}")
                # Then remove the directory
                os.rmdir(persistent_uploads_dir)
            except Exception:
                # Fallback to rmtree if individual removal fails
                shutil.rmtree(persistent_uploads_dir)
            logger.info(f"Cleaned up persistent uploads directory: {persistent_uploads_dir}")
        else:
            logger.debug(f"Persistent uploads directory does not exist: {persistent_uploads_dir}")
    except Exception as e:
        logger.warning(f"Error cleaning up persistent uploads directory: {e}", exc_info=True)
        # Don't raise - cleanup failures shouldn't break the flow

def create_error_images_zip(rows_with_paths: List[Dict[str, Any]], error_type: str) -> Optional[str]:
    """
    Create a zip file containing images from error rows (FN or FP).
    
    Args:
        rows_with_paths: List of row dictionaries with 'file_path' field
        error_type: 'fn' or 'fp' for naming the zip file
    
    Returns:
        Path to the created zip file, or None if no files to zip
    """
    if not rows_with_paths or len(rows_with_paths) == 0:
        return None
    
    try:
        # Create a temporary zip file
        temp_dir = tempfile.gettempdir()
        timestamp = int(time.time())
        zip_filename = f"{error_type}_images_{timestamp}.zip"
        zip_path = os.path.join(temp_dir, zip_filename)
        
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            added_count = 0
            for row in rows_with_paths:
                file_path = row.get('file_path') or row.get('file')
                if not file_path:
                    continue
                
                # Handle URLs (download first)
                if file_path.startswith('http://') or file_path.startswith('https://'):
                    try:
                        # Download the image temporarily
                        response = requests.get(file_path, timeout=30)
                        response.raise_for_status()
                        img_data = response.content
                        
                        # Determine filename from URL or use default
                        filename_from_url = os.path.basename(file_path.split('?')[0])  # Remove query params
                        if not filename_from_url or '.' not in filename_from_url:
                            filename_from_url = f"image_{added_count}.jpg"
                        
                        # Add metadata to filename if available
                        if 'gt' in row and 'pred' in row:
                            name, ext = os.path.splitext(filename_from_url)
                            filename_in_zip = f"{name}_GT{row['gt']}_PRED{row['pred']}{ext}"
                        else:
                            filename_in_zip = filename_from_url
                        
                        zipf.writestr(filename_in_zip, img_data)
                        added_count += 1
                    except Exception as e:
                        logger.warning(f"Could not download image from URL {file_path}: {e}")
                        continue
                else:
                    # Handle local file paths
                    # Handle both full paths and basenames
                    if not os.path.isabs(file_path):
                        # Try to find the file in common locations
                        possible_paths = [
                            file_path,
                            os.path.join('./persistent_uploads', file_path),
                            os.path.join('.', file_path),
                        ]
                        found = False
                        for possible in possible_paths:
                            if os.path.exists(possible) and os.path.isfile(possible):
                                file_path = possible
                                found = True
                                break
                        if not found:
                            logger.warning(f"Could not find file: {row.get('file', 'unknown')}")
                            continue
                    
                    if os.path.exists(file_path) and os.path.isfile(file_path):
                        # Use the original filename in the zip
                        filename_in_zip = os.path.basename(file_path)
                        # Add metadata to filename if available
                        if 'gt' in row and 'pred' in row:
                            name, ext = os.path.splitext(filename_in_zip)
                            filename_in_zip = f"{name}_GT{row['gt']}_PRED{row['pred']}{ext}"
                        
                        zipf.write(file_path, filename_in_zip)
                        added_count += 1
                    else:
                        logger.warning(f"File not found: {file_path}")
        
        if added_count == 0:
            if os.path.exists(zip_path):
                os.remove(zip_path)
            return None
        
        logger.info(f"Created {error_type.upper()} zip file: {zip_path} with {added_count} images")
        return zip_path
    except Exception as e:
        logger.error(f"Error creating {error_type} zip file: {e}", exc_info=True)
        return None

def load_market_photos_preview(market: str):
    """Load and preview photos from CSV for a given market"""
    if not market or not market.strip():
        return "❌ Please enter a market name first."
    
    market = market.strip()
    logger.info(f"Loading preview for market: {market}")
    
    base_photos, error_msg = load_base_photos_from_csv(market)
    
    if error_msg:
        return f"❌ {error_msg}"
    
    if not base_photos:
        return f"⚠️ No photos found for market '{market}'."
    
    # Count PASS and FAIL
    pass_count = sum(1 for p in base_photos if p['evaluation_status'] == 'PASS')
    fail_count = sum(1 for p in base_photos if p['evaluation_status'] == 'FAIL')
    
    return f"✅ Found {len(base_photos)} photos for market '{market}': {pass_count} PASS, {fail_count} FAIL"


def _extract_upright_from_checklist(visual_checklist: Dict[str, Any]) -> Optional[bool]:
    """Infer upright/not-upright from visual_checklist fields."""
    if not isinstance(visual_checklist, dict):
        return None

    direct_flag = visual_checklist.get("is_vehicle_upright")
    if isinstance(direct_flag, bool):
        return direct_flag

    posture_text = str(visual_checklist.get("vehicle_posture", "")).strip().lower()
    if not posture_text:
        return None

    not_upright_tokens = ["tipped", "fallen", "lying", "on its side", "collapsed", "not upright"]
    upright_tokens = ["upright", "standing", "kickstand", "slightly tilted", "leaning"]

    if any(token in posture_text for token in not_upright_tokens):
        return False
    if any(token in posture_text for token in upright_tokens):
        return True
    return None


def extract_hallucination_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Return rows where checklist posture conflicts with final decision.
    Smoking-gun examples:
    - checklist says upright, decision is FAIL
    - checklist says not upright, decision is PASS
    """
    hallucinations: List[Dict[str, Any]] = []

    for row in rows:
        visual_checklist = row.get("visual_checklist", {}) or {}
        upright_state = _extract_upright_from_checklist(visual_checklist)
        pred = str(row.get("pred", "")).upper().strip()

        issue = None
        if upright_state is True and pred == "FAIL":
            issue = "Checklist says upright, but decision is FAIL"
        elif upright_state is False and pred == "PASS":
            issue = "Checklist says not upright/tipped, but decision is PASS"

        if issue:
            enriched = dict(row)
            enriched["hallucination_issue"] = issue
            enriched["checklist_vehicle_posture"] = visual_checklist.get("vehicle_posture", "")
            hallucinations.append(enriched)

    return hallucinations


def build_hallucination_gallery_items(hall_rows: List[Dict[str, Any]]) -> List[Tuple[str, str]]:
    """Create Gradio gallery items as (image_path_or_url, caption)."""
    gallery_items: List[Tuple[str, str]] = []
    for row in hall_rows:
        img = row.get("file_path") or row.get("file")
        if not img:
            continue
        caption = f"{row.get('file', 'image')} | pred={row.get('pred', '')} | {row.get('hallucination_issue', '')}"
        gallery_items.append((img, caption))
    return gallery_items


def _safe_int(value: Any, default: int = 0) -> int:
    try:
        return int(float(value))
    except Exception:
        return default


def _compute_fpr_pct(fp: int, tn: int) -> float:
    denom = fp + tn
    return round((fp / denom) * 100.0, 2) if denom > 0 else 0.0


def run_optimizer_wrapper(files, market, ux_focus_pct, evaluation_only=False, system_prompt="", user_prompt=""):
    """Wrapper to provide status updates"""
    wrapper_start_time = time.time()
    # Reset stop event at the very start (before any processing)
    stop_event.clear()
    
    # Set evaluation_only mode if requested from UI
    if evaluation_only:
        global EVALUATION_ONLY
        EVALUATION_ONLY = True
        logger.info("📊 Evaluation-only mode enabled from UI")
    
    # Clean up persistent uploads directory at START of each run
    # This ensures fresh state for each batch upload
    # Note: Small delay to ensure previous uploads are fully closed
    import time as time_module
    time_module.sleep(0.5)  # Small delay to let previous uploads finish
    cleanup_persistent_uploads()
    
    logger.info("=" * 80)
    logger.info("run_optimizer_wrapper called")
    logger.info(f"Files type: {type(files)}, value: {files}")
    logger.info(f"Market: {market}, UX Focus: {ux_focus_pct}, Evaluation Only: {evaluation_only}")
    logger.info("Stop event cleared in wrapper")
    
    # Check if files were actually received (Gradio might pass None on upload failure)
    if files is None:
        logger.warning("⚠️ Files parameter is None - upload may have failed")
    elif isinstance(files, (list, tuple)) and len(files) == 0:
        logger.info("Files list is empty (no files uploaded)")
    else:
        # Log file info to help diagnose upload issues
        if isinstance(files, (list, tuple)):
            file_sizes = []
            for f in files:
                if f and os.path.exists(f):
                    try:
                        size_mb = os.path.getsize(f) / (1024 * 1024)
                        file_sizes.append(size_mb)
                    except:
                        pass
            if file_sizes:
                total_mb = sum(file_sizes)
                avg_mb = total_mb / len(file_sizes)
                max_mb = max(file_sizes)
                logger.info(f"Uploaded {len(files)} files: total={total_mb:.2f}MB, avg={avg_mb:.2f}MB, max={max_mb:.2f}MB")
                if total_mb > 100:
                    logger.warning(f"⚠️ Large upload: {total_mb:.2f}MB total - this may take time")
    
    logger.info(f"Time since wrapper start: {time.time() - wrapper_start_time:.3f}s")
    
    empty_df = pd.DataFrame()
    
    # Handle None or empty files - Gradio might pass None
    if files is None:
        files = []
    elif not isinstance(files, (list, tuple)):
        # Gradio might pass a single file path as a string
        files = [files] if files else []
    
    # Copy uploaded files to persistent directory AND convert to data URLs immediately
    # This prevents Gradio from deleting them while CSV processing happens
    persistent_uploads_dir = "./persistent_uploads"
    processed_files = None  # Will contain list of dicts with file path and pre-computed data_url
    if files:
        try:
            # Create persistent uploads directory if it doesn't exist
            os.makedirs(persistent_uploads_dir, exist_ok=True)
            logger.info(f"Processing {len(files)} uploaded files: copying to persistent directory and converting to data URLs...")
            processed_files = []  # Initialize list for processed files
            
            # Log file sizes first to identify large files
            file_sizes = []
            for temp_path in files:
                if temp_path and os.path.exists(temp_path):
                    try:
                        size_mb = os.path.getsize(temp_path) / (1024 * 1024)
                        file_sizes.append(size_mb)
                    except:
                        pass
            if file_sizes:
                total_mb = sum(file_sizes)
                max_mb = max(file_sizes)
                avg_mb = total_mb / len(file_sizes)
                logger.info(f"File sizes: total={total_mb:.2f}MB, avg={avg_mb:.2f}MB, max={max_mb:.2f}MB")
                if max_mb > 5:
                    logger.warning(f"⚠️ Large files detected (max={max_mb:.2f}MB) - compression may take time")
            
            processing_start = time.time()
            
            # Process files in parallel for much better performance
            def process_single_file(args):
                idx, temp_path = args
                try:
                    if not temp_path or not os.path.exists(temp_path):
                        logger.warning(f"Skipping non-existent file: {temp_path}")
                        return None
                    
                    # Generate a unique filename to avoid collisions
                    original_name = os.path.basename(temp_path)
                    # Preserve PASS_/FAIL_ prefix in the filename
                    persistent_path = os.path.join(persistent_uploads_dir, original_name)
                    
                    # If file already exists, add a counter (need to check atomically)
                    counter = 1
                    base_name, ext = os.path.splitext(original_name)
                    while os.path.exists(persistent_path):
                        persistent_path = os.path.join(persistent_uploads_dir, f"{base_name}_{counter}{ext}")
                        counter += 1
                    
                    # Copy file to persistent location
                    shutil.copy2(temp_path, persistent_path)
                    logger.debug(f"Copied {original_name} to {persistent_path}")
                    
                    # IMMEDIATELY convert to data URL and store in memory (before CSV processing)
                    file_start = time.time()
                    try:
                        data_url = to_data_url(persistent_path)
                        file_elapsed = time.time() - file_start
                        if file_elapsed > 2.0:  # Log slow conversions
                            logger.warning(f"⚠️ Slow conversion: {original_name} took {file_elapsed:.2f}s")
                        else:
                            logger.debug(f"Converted {original_name} to data URL in {file_elapsed:.2f}s")
                        
                        return {
                            "file": persistent_path,
                            "data_url": data_url,  # Pre-computed and held in memory
                            "original_name": original_name
                        }
                    except Exception as e:
                        logger.warning(f"Error converting {original_name} to data URL: {e}")
                        # Still return the file path, but data_url will be computed later
                        return {
                            "file": persistent_path,
                            "data_url": "",  # Will be computed later if needed
                            "original_name": original_name
                        }
                except Exception as e:
                    logger.error(f"Error processing file {temp_path}: {e}")
                    return None
            
            # Process all files in parallel
            with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(files))) as executor:
                results = list(executor.map(process_single_file, enumerate(files, 1)))
            
            # Filter out None results and add to processed_files
            for result in results:
                if result:
                    processed_files.append(result)
            
            processing_elapsed = time.time() - processing_start
            logger.info(f"Successfully processed {len(processed_files)}/{len(files)} files in {processing_elapsed:.2f}s (copied and converted to data URLs)")
            if processing_elapsed > 30:
                logger.warning(f"⚠️ File processing took {processing_elapsed:.2f}s - consider using smaller files or market photos")
            # Store processed files for later use in run_optimizer
            # We'll pass this as a special parameter or store it in a way run_optimizer can access
        except Exception as e:
            logger.error(f"Error processing uploaded files: {e}", exc_info=True)
            # Clean up on error to prevent issues with next batch
            try:
                cleanup_persistent_uploads()
            except:
                pass  # Ignore cleanup errors during error handling
            # Continue with original files if processing fails (fallback)
            logger.warning("Continuing with original file paths (may be cleaned up by Gradio)")
            processed_files = None
    
    # Check if we have either files or market
    if not files and (not market or not market.strip()):
        logger.warning("No files and no market provided")
        return (
            "❌ Error: Please upload images (filenames must start with PASS_ or FAIL_) or enter a market name.",
            "", 
            empty_df, 
            empty_df, 
            empty_df,
            empty_df,
            "❌ No files uploaded and no market specified",
            0.0,
            "None",  # market_display
            "",  # photo_stats
            [],
            [],
            [],
        )
    
    try:
        # Update status at start
        files_count = len(files) if files else 0
        market_text = f" for market '{market.strip()}'" if market and market.strip() else ""
        status_msg = f"🔄 Running... Processing {files_count} uploaded files{market_text} (UX Focus: {ux_focus_pct}%)"
        logger.info(status_msg)
        logger.info(f"About to call run_optimizer with {files_count} files")
        
        # Run the optimizer
        optimizer_call_start = time.time()
        logger.info(f"Calling run_optimizer... (time since wrapper start: {optimizer_call_start - wrapper_start_time:.3f}s)")
        try:
            summary, best_prompt, df_history, df_fn, df_fp, df_hall, market_display, photo_stats, fn_rows_with_paths, fp_rows_with_paths, hall_rows_with_paths = run_optimizer(
                files if processed_files is None else [p["file"] for p in processed_files],  # Use persistent paths
                market=market.strip() if market else "",
                ux_focus_pct=ux_focus_pct,
                progress=gr.Progress(track_tqdm=False),
                preprocessed_uploaded_files=processed_files,  # Pass pre-processed files with data URLs
                custom_system_prompt=system_prompt,
                custom_user_prompt=user_prompt
            )
            optimizer_call_end = time.time()
            logger.info(f"run_optimizer returned successfully (took {optimizer_call_end - optimizer_call_start:.2f}s)")
        except Exception as e:
            optimizer_call_end = time.time()
            logger.error(f"Error in run_optimizer call after {optimizer_call_end - optimizer_call_start:.2f}s: {type(e).__name__}: {e}", exc_info=True)
            raise
        logger.info(f"Received: summary type={type(summary)}, prompt type={type(best_prompt)}")
        logger.info(f"Received: df_history type={type(df_history)}, shape={df_history.shape if hasattr(df_history, 'shape') else 'N/A'}")
        logger.info(f"Received: df_fn type={type(df_fn)}, shape={df_fn.shape if hasattr(df_fn, 'shape') else 'N/A'}")
        logger.info(f"Received: df_fp type={type(df_fp)}, shape={df_fp.shape if hasattr(df_fp, 'shape') else 'N/A'}")
        
        # Success status
        if evaluation_only:
            status_msg = "✅ Evaluation completed successfully! (Baseline only - no optimization iterations)"
        else:
            status_msg = "✅ Optimization completed successfully!"
        logger.info(status_msg)
        logger.info("Returning results to Gradio UI...")
        
        # NOTE: Do NOT clean up persistent uploads here - it will be done in run_optimizer_with_downloads
        # after zip files are created. Cleaning up here would delete files before zip creation.
        
        return summary, best_prompt, df_history, df_fn, df_fp, df_hall, status_msg, 1.0, market_display, photo_stats, fn_rows_with_paths, fp_rows_with_paths, hall_rows_with_paths
        
    except InterruptedError as e:
        status_msg = f"⚠️ Optimization stopped by user: {str(e)}"
        logger.warning(status_msg)
        # Clear stop event for next run
        stop_event.clear()
        # Clean up persistent uploads directory even on interruption
        cleanup_persistent_uploads()
        empty_df = pd.DataFrame()
        return (
            status_msg,
            "",
            empty_df,
            empty_df,
            empty_df,
            empty_df,
            status_msg,
            0.0,
            "",
            "",
            [],
            [],
            []
        )
    except Exception as e:
        error_type = type(e).__name__
        error_str = str(e)
        error_msg = f"❌ Error: {error_type}: {error_str}"
        
        # Provide helpful message for common errors
        if "ClientDisconnect" in error_type or "disconnect" in error_str.lower():
            error_msg = "❌ Upload Error: Connection lost during file upload. This can happen with large files or slow networks.\n\n**Suggestions:**\n- Try uploading fewer files at once (5-10 at a time)\n- Use market photos instead (enter market name)\n- Check your network connection\n- Try again with smaller batches\n- If files are >5MB each, consider compressing them first"
        elif "timeout" in error_str.lower() or "timed out" in error_str.lower():
            error_msg = f"❌ Timeout Error: {error_str}\n\n**Suggestions:**\n- Try with fewer files (5-10 at a time)\n- Check your network connection\n- Use market photos instead (enter market name)\n- Large files (>5MB each) may timeout - try compressing them"
        elif "FileNotFoundError" in error_type or "No such file" in error_str:
            error_msg = f"❌ File Error: Files not found after upload. This may indicate an upload failure.\n\n**Suggestions:**\n- Try uploading again\n- Upload fewer files at once\n- Check that files are valid images\n- Use market photos instead"
        elif "PermissionError" in error_type or "permission" in error_str.lower():
            error_msg = f"❌ Permission Error: Cannot access uploaded files.\n\n**Suggestions:**\n- Try uploading again\n- Check file permissions\n- Use market photos instead"
        
        logger.error(error_msg, exc_info=True)
        # Clear stop event even on error so next run can proceed
        stop_event.clear()
        # Clean up persistent uploads directory even on error
        cleanup_persistent_uploads()
        empty_df = pd.DataFrame()
        return (
            error_msg,
            "",
            empty_df,
            empty_df,
            empty_df,
            empty_df,
            error_msg,
            0.0,
            "",
            "",
            [],
            [],
            []
        )

with gr.Blocks(title="Prompt Evaluator & Comparator") as demo:
    gr.Markdown("## Prompt Comparison & Evaluation")
    
    with gr.Row():
        with gr.Column(scale=2):
            with gr.Row():
                market_input = gr.Textbox(
                    label="Market Name (optional)",
                    placeholder="Enter market name (e.g., Denver, Rome) to load base photos from CSV",
                    info="Case-insensitive. Loads photos from 'ai photo review - examples.csv'",
                    scale=4
                )
                load_market_btn = gr.Button("Load Photos", variant="secondary", scale=1)
            market_preview = gr.Textbox(
                label="Market Preview",
                value="",
                interactive=False,
                lines=2,
                visible=True
            )
            gr.Markdown(
                "**⚠️ IMPORTANT - File Upload Issues:**\n"
                "If you see a 'network connection' or 'ClientDisconnect' error, this is a known Gradio limitation.\n\n"
                "**✅ RECOMMENDED SOLUTION:** Use market photos instead (much more reliable!):\n"
                "1. Enter a market name above (e.g., 'Atlanta', 'Denver')\n"
                "2. Click 'Load Photos' to preview\n"
                "3. Click 'Run evaluation' - no file upload needed!\n\n"
                "**If you must upload files:**\n"
                "- Upload **3-5 files at a time** (not 10+)\n"
                "- **Wait for first batch to complete** before uploading second batch\n"
                "- Ensure **stable network connection**\n"
                "- Files should be **<5MB each** (compress if larger)\n"
                "- **If second batch fails:** Refresh the page and try again\n"
                "- **Best practice:** Use market photos for reliability"
            )
            files = gr.File(
                file_count="multiple",
                type="filepath",
                label="Upload images (optional, filenames must start with PASS_ or FAIL_)",
                file_types=["image"],
            )
        with gr.Column(scale=1):
            ux_focus_slider = gr.Slider(
                minimum=0,
                maximum=100,
                value=0,
                step=1,
                label="UX Focus (%) - 0% = GP focused (maximize recall, minimize FN) | 100% = UX focused (maximize precision, minimize FP)"
            )
            evaluation_only_checkbox = gr.Checkbox(
                label="Evaluation Only (skip optimization)",
                value=True,
                info="If checked, runs only baseline evaluation (iteration 0) and returns recall/precision results without optimization iterations"
            )
            market_display = gr.Textbox(
                label="Market",
                value="None",
                interactive=False,
                lines=1
            )
            photo_stats_display = gr.Textbox(
                label="Photo Statistics",
                value="",
                interactive=False,
                lines=2
            )
            status_box = gr.Textbox(
                label="Status",
                value="⏸️ Ready - Enter market name and/or upload images, then click 'Run evaluation'",
                interactive=False,
                lines=3
            )
            progress_bar = gr.Slider(
                minimum=0,
                maximum=1,
                value=0,
                step=0.01,
                label="Progress",
                visible=True,
                interactive=False
            )

    # Prompts section
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Base Prompts (Optional - leave empty to use defaults)")
            with gr.Row():
                load_defaults_btn = gr.Button("Load Default Prompts", variant="secondary")
            system_prompt_input = gr.Textbox(
                label="System Prompt",
                value="",
                placeholder="Leave empty to use default system prompt. You can paste text here (Ctrl+V / Cmd+V).",
                lines=6,
                interactive=True
            )
            user_prompt_input = gr.Textbox(
                label="User Prompt (Base)",
                value="",
                placeholder="Leave empty to use default user prompt. You can paste text here (Ctrl+V / Cmd+V).",
                lines=10,
                interactive=True
            )
    
    # Load defaults button handler
    def load_default_prompts():
        try:
            default_system = DEFAULT_SYSTEM_PROMPT if 'DEFAULT_SYSTEM_PROMPT' in globals() else ""
            default_user = DEFAULT_BASE_USER_PROMPT if 'DEFAULT_BASE_USER_PROMPT' in globals() else ""
            return default_system, default_user
        except:
            return "", ""
    
    load_defaults_btn.click(
        fn=load_default_prompts,
        inputs=[],
        outputs=[system_prompt_input, user_prompt_input]
    )

    with gr.Row():
        run_btn = gr.Button("Run evaluation", variant="primary")
        stop_btn = gr.Button("Stop", variant="stop")

    with gr.Tabs():
        with gr.Tab("Evaluation Results"):
            summary = gr.Textbox(lines=8, label="Summary", interactive=False)
            best_prompt = gr.Textbox(lines=12, label="Best user prompt", interactive=False)
            history_df = gr.Dataframe(label="Iteration history (percentages + counts)")
            
            # FN/FP tables with download buttons (stacked vertically)
            fn_df = gr.Dataframe(label="False negatives (GT PASS, Pred FAIL)")
            fn_csv_download = gr.File(label="📥 Download False Negatives as CSV", visible=False)
            fp_df = gr.Dataframe(label="False positives (GT FAIL, Pred PASS)")
            fp_csv_download = gr.File(label="📥 Download False Positives as CSV", visible=False)
            
            # Download buttons for FN/FP images
            with gr.Row():
                fn_download_btn = gr.File(label="📥 Download False Negatives Images (ZIP)", visible=True)
                fp_download_btn = gr.File(label="📥 Download False Positives Images (ZIP)", visible=True)

        with gr.Tab("Hallucination Gallery"):
            hallucination_summary = gr.Textbox(
                label="Hallucination Summary",
                value="Run an evaluation to populate checklist-vs-decision mismatches.",
                interactive=False,
                lines=3
            )
            hall_df = gr.Dataframe(label="Checklist/Decision Mismatches")
            hall_gallery = gr.Gallery(label="Smoking Gun Images", columns=4, height=360, object_fit="contain")

    gr.Markdown("### Prompt Comparison (A/B) — Side-by-Side Safety Check")
    with gr.Row():
        prompt_a_input = gr.Textbox(
            label="Prompt A (Current)",
            value="",
            placeholder="Paste current prompt (Prompt A)",
            lines=8,
            interactive=True
        )
        prompt_b_input = gr.Textbox(
            label="Prompt B (Candidate)",
            value="",
            placeholder="Paste candidate prompt (Prompt B)",
            lines=8,
            interactive=True
        )
    compare_btn = gr.Button("Compare Prompt A vs B", variant="primary")
    compare_summary = gr.Textbox(label="Comparison Summary", lines=6, interactive=False)
    compare_metrics_df = gr.Dataframe(label="A/B Metrics Comparison")
    with gr.Row():
        compare_fp_a_df = gr.Dataframe(label="Prompt A False Positives")
        compare_fp_b_df = gr.Dataframe(label="Prompt B False Positives")

    # Load market photos preview button
    load_market_btn.click(
        fn=load_market_photos_preview,
        inputs=[market_input],
        outputs=[market_preview],
    )
    
    def run_optimizer_with_downloads(files, market, ux_focus_pct, evaluation_only, system_prompt, user_prompt):
        """Wrapper that creates zip files and CSV downloads"""
        try:
            result = run_optimizer_wrapper(files, market, ux_focus_pct, evaluation_only, system_prompt, user_prompt)
            summary, best_prompt, df_history, df_fn, df_fp, df_hall, status_msg, progress_val, market_display, photo_stats, fn_rows, fp_rows, hall_rows = result
            
            # Create zip files for image downloads (BEFORE cleanup, so files still exist)
            fn_zip = create_error_images_zip(fn_rows, 'fn') if fn_rows and len(fn_rows) > 0 else None
            fp_zip = create_error_images_zip(fp_rows, 'fp') if fp_rows and len(fp_rows) > 0 else None
            
            # Create CSV files for table downloads
            fn_csv = None
            fp_csv = None
            if len(df_fn) > 0:
                try:
                    temp_dir = tempfile.gettempdir()
                    timestamp = int(time.time())
                    fn_csv_path = os.path.join(temp_dir, f"false_negatives_{timestamp}.csv")
                    df_fn.to_csv(fn_csv_path, index=False)
                    fn_csv = fn_csv_path
                    logger.info(f"Created FN CSV: {fn_csv_path}")
                except Exception as e:
                    logger.error(f"Error creating FN CSV: {e}")
            
            if len(df_fp) > 0:
                try:
                    temp_dir = tempfile.gettempdir()
                    timestamp = int(time.time())
                    fp_csv_path = os.path.join(temp_dir, f"false_positives_{timestamp}.csv")
                    df_fp.to_csv(fp_csv_path, index=False)
                    fp_csv = fp_csv_path
                    logger.info(f"Created FP CSV: {fp_csv_path}")
                except Exception as e:
                    logger.error(f"Error creating FP CSV: {e}")
            
            hall_gallery_items = build_hallucination_gallery_items(hall_rows)
            hall_summary = (
                f"Hallucination rows: {len(hall_rows)}\n"
                f"Rule: checklist says upright + decision FAIL OR checklist says not-upright + decision PASS"
            )

            return (
            summary, 
            best_prompt, 
            df_history, 
            df_fn, 
            df_fp,
            df_hall,
            hall_summary,
            hall_gallery_items,
            status_msg, 
            progress_val, 
            market_display, 
            photo_stats,
            gr.update(value=fn_zip, visible=fn_zip is not None) if fn_zip else gr.update(visible=False),
            gr.update(value=fp_zip, visible=fp_zip is not None) if fp_zip else gr.update(visible=False),
            gr.update(value=fn_csv, visible=fn_csv is not None) if fn_csv else gr.update(visible=False),
            gr.update(value=fp_csv, visible=fp_csv is not None) if fp_csv else gr.update(visible=False)
        )
        finally:
            # Always clean up persistent uploads directory AFTER creating zip files (or if zip creation fails)
            cleanup_persistent_uploads()

    def run_prompt_comparison(files, market, system_prompt, prompt_a, prompt_b):
        """Run side-by-side Prompt A vs Prompt B comparison on the same dataset."""
        if not (prompt_a and prompt_a.strip()) or not (prompt_b and prompt_b.strip()):
            empty_df = pd.DataFrame()
            return (
                "❌ Please provide both Prompt A and Prompt B.",
                empty_df,
                empty_df,
                empty_df,
            )

        # Run both prompts in evaluation-only mode for stable A/B comparison
        prev_eval_only = EVALUATION_ONLY if 'EVALUATION_ONLY' in globals() else False
        try:
            a_result = run_optimizer_wrapper(files, market, 0, True, system_prompt, prompt_a.strip())
            b_result = run_optimizer_wrapper(files, market, 0, True, system_prompt, prompt_b.strip())
        finally:
            if 'EVALUATION_ONLY' in globals():
                globals()['EVALUATION_ONLY'] = prev_eval_only

        (_, _, a_hist, _a_fn, a_fp, _a_hall, a_status, _a_prog, _a_market, _a_stats, _a_fn_rows, _a_fp_rows, _a_hall_rows) = a_result
        (_, _, b_hist, _b_fn, b_fp, _b_hall, b_status, _b_prog, _b_market, _b_stats, _b_fn_rows, _b_fp_rows, _b_hall_rows) = b_result

        if len(a_hist) == 0 or len(b_hist) == 0:
            empty_df = pd.DataFrame()
            return (
                f"❌ Comparison failed. A: {a_status} | B: {b_status}",
                empty_df,
                a_fp if isinstance(a_fp, pd.DataFrame) else empty_df,
                b_fp if isinstance(b_fp, pd.DataFrame) else empty_df,
            )

        a_row = a_hist.iloc[0]
        b_row = b_hist.iloc[0]

        a_fp_count = _safe_int(a_row.get("fp", 0))
        a_tn_count = _safe_int(a_row.get("tn", 0))
        b_fp_count = _safe_int(b_row.get("fp", 0))
        b_tn_count = _safe_int(b_row.get("tn", 0))

        a_fpr = _compute_fpr_pct(a_fp_count, a_tn_count)
        b_fpr = _compute_fpr_pct(b_fp_count, b_tn_count)
        fpr_delta = round(b_fpr - a_fpr, 2)

        compare_df = pd.DataFrame([
            {
                "prompt": "A",
                "fpr_pct": a_fpr,
                "fp": a_fp_count,
                "tn": a_tn_count,
                "recall_pass_pct": float(a_row.get("recall_pass_pct", 0)),
                "precision_pass_pct": float(a_row.get("precision_pass_pct", 0)),
                "logic_mismatch_pct": float(a_row.get("logic_mismatch_pct", 0) if "logic_mismatch_pct" in a_row else 0),
            },
            {
                "prompt": "B",
                "fpr_pct": b_fpr,
                "fp": b_fp_count,
                "tn": b_tn_count,
                "recall_pass_pct": float(b_row.get("recall_pass_pct", 0)),
                "precision_pass_pct": float(b_row.get("precision_pass_pct", 0)),
                "logic_mismatch_pct": float(b_row.get("logic_mismatch_pct", 0) if "logic_mismatch_pct" in b_row else 0),
            },
        ])

        safer = "Prompt A"
        if b_fpr < a_fpr:
            safer = "Prompt B"
        elif b_fpr == a_fpr:
            safer = "Tie"

        summary_text = (
            f"FPR comparison (lower is safer): A={a_fpr}% vs B={b_fpr}% | Δ(B-A)={fpr_delta}%\n"
            f"Safer prompt by FPR: {safer}\n"
            f"A precision={a_row.get('precision_pass_pct', 0)}%, B precision={b_row.get('precision_pass_pct', 0)}%"
        )

        return summary_text, compare_df, a_fp, b_fp

    compare_btn.click(
        fn=run_prompt_comparison,
        inputs=[files, market_input, system_prompt_input, prompt_a_input, prompt_b_input],
        outputs=[compare_summary, compare_metrics_df, compare_fp_a_df, compare_fp_b_df],
    )
    
    run_btn.click(
        fn=run_optimizer_with_downloads,
        inputs=[files, market_input, ux_focus_slider, evaluation_only_checkbox, system_prompt_input, user_prompt_input],
        outputs=[summary, best_prompt, history_df, fn_df, fp_df, hall_df, hallucination_summary, hall_gallery, status_box, progress_bar, market_display, photo_stats_display, fn_download_btn, fp_download_btn, fn_csv_download, fp_csv_download],
    )
    
    stop_btn.click(
        fn=stop_optimization,
        inputs=[],
        outputs=[status_box],
    )


if __name__ == "__main__":
    # Note: Gradio handles file uploads automatically
    # For slow uploads, users should upload smaller batches (5-10 files at a time)
    demo.launch(share=True)


/Users/assafkazakov/projects/playground/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-16 18:10:29,099 - INFO - ✅ Loaded 2437 cached responses from disk
2026-03-16 18:10:30,096 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-16 18:10:30,276 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-16 18:10:30,284 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860


2026-03-16 18:10:31,209 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-03-16 18:10:31,552 - INFO - HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_darwin_arm64 "HTTP/1.1 200 OK"



Could not create share link. Missing file: /Users/assafkazakov/.cache/huggingface/gradio/frpc/frpc_darwin_arm64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_darwin_arm64
2. Rename the downloaded file to: frpc_darwin_arm64_v0.3
3. Move the file to this location: /Users/assafkazakov/.cache/huggingface/gradio/frpc
